# Notebook 12 — StratLake Campaign Evidence Gap and Promotion Readiness Review

This is a raw pre-import research/development draft for `christophermoverton/fintech-stratlake-notebook-workflows`.

**Theme:** From isolated expanded runs to campaign-level evidence review.

Notebook 12 is a campaign evidence reviewer. It moves beyond Notebook 11's isolated expanded-run evidence review and asks whether a structured StratLake research campaign produced enough artifact-backed evidence for further human review.

**Future committed path:** `notebooks/12_stratlake_campaign_evidence_gap_promotion_readiness.ipynb`  
**Future milestone:** `M15 — Notebook 12 Campaign Evidence Gap and Promotion Readiness Review`  
**Future branch:** `features/m15-notebook-12-campaign-evidence-gap-promotion-readiness`

## Core question

Across a structured StratLake research campaign, which strategies, windows, configs, runs, and artifacts produced enough evidence to justify further human review?

More specifically:

Can campaign-level outputs connect strategy candidates, run IDs, windows, metrics, split metrics, promotion gates, manifests, failures, retries, reuse, and caveats into a coherent evidence review table?


## Source-safe and native-command-first posture

`stratlake-trade-engine` remains the source of truth for campaign execution, strategy execution, metrics, split metrics, promotion gates, governance reports, artifact manifests, retry/reuse/checkpoint records, and campaign reports.

This notebook only orchestrates, restores, discovers, loads, reviews, classifies, summarizes, and hands off.

It does **not** reimplement StratLake strategy logic, campaign orchestration, metrics, promotion gates, or governance logic.

## Explicit non-claims

This raw draft makes the following non-claims:

- `no_strategy_approval_claim`
- `no_alpha_claim`
- `no_production_readiness_claim`
- `no_statistical_significance_claim`
- `no_promotion_grade_claim`
- `no_complete_platform_artifact_claim_unless_verified`
- `no_ci_runtime_equivalence_claim`
- `no_campaign_correctness_claim_without_native_artifacts`

Statuses such as `ready_for_human_watchlist_review` mean only that a row may have enough platform-backed evidence for human review. They do not mean approval, promotion, alpha confirmation, production readiness, or statistical significance.


## Relationship to Notebook 10 and Notebook 11

Notebook 10 reviewed smoke-mode strategy evidence and promotion-readiness caveats.

Notebook 11 validated a guarded expanded-run smoke path but found that complete promotion evidence remained incomplete without platform-backed split metrics, promotion gates, and complete review artifacts.

Notebook 12 builds from that stance without simply repeating manual expanded runs. It searches for campaign-level evidence surfaces and asks whether campaign artifacts can support a conservative evidence review table.


## Notebook 11 carry-forward context

Notebook 11's prior raw/runtime stance is carried forward as context, not as campaign evidence.

Prior final stance:

```text
notebook_11_import_pr_ready
```

Prior runtime-smoke stance:

```text
notebook_11_expanded_run_smoke_passed_with_metrics_review_artifacts_incomplete
```

Successful expanded-run smoke metrics:

```text
expanded_runs_attempted = 4
expanded_runs_completed = 4
expanded_runs_failed = 0
expanded_metric_rows = 4
expanded_artifact_metric_rows = 4
expanded_stdout_metric_rows = 0
manual_review_skipped_count = 0
preview_only_execution_rows = 0
```

Manual-review candidates:

```text
buy_and_hold_v1
cross_section_momentum
seeded_random_v1
sma_crossover_v1
```

Remaining blockers:

```text
expanded_split_metric_rows = 0
expanded_platform_promotion_gates_loaded_count = 0
expanded_promotion_gates_loaded_count = 0
expanded_complete_review_artifact_count = 0
notebook11_interpretive_package_incomplete_platform_count = 4
platform_review_artifacts_required_for_complete_promotion_evidence = true
promotion_grade_claim_made = false
```

Notebook 12 should use this as a bridge from expanded-run evidence to campaign evidence. It should not convert these four strategy candidates into campaign rows without a platform campaign registry or clear run/campaign mapping.


## 1. Install notebook dependencies and app packages

This install cell follows the established Notebook 10/11 TestPyPI + PyPI fallback pattern. It is intentionally kept as a notebook cell for future Colab use and should remain source-safe.


In [ ]:
!pip install -q "pandas-market-calendars>=5.0"
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ fintech-market-ingestion
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ stratlake-trade-engine


## 2. Imports, Colab detection, and display helpers

These helpers avoid secrets, avoid fabricated evidence, and treat missing optional packages or artifacts as caveats rather than notebook crashes.


In [ ]:
import importlib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None

try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    userdata = None
    IN_COLAB = False


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def show_df(df: pd.DataFrame, n: int = 20) -> None:
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n).to_string(index=False))


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_json_dumps(obj: Any) -> str:
    return json.dumps(obj, indent=2, sort_keys=True, default=str)


def print_json(obj: Any) -> None:
    print(safe_json_dumps(obj))


## 3. Runtime controls and smoke-test profiles

Use `NOTEBOOK12_TEST_PROFILE` to switch common smoke-test postures without editing many booleans by hand.

Recommended profiles:

```text
cold_smoke_1_preview
cold_smoke_2_loose_restored_discovery
cold_smoke_3_strict_negative_control
cold_smoke_4_strict_missing_filter_guardrail
cold_smoke_5_command_shape_readiness
campaign_smoke_preview
campaign_smoke_dry_run
campaign_smoke_dry_run_allow_provisional
campaign_smoke_execute_allow_provisional_no_dry_run
custom
```

Default is `cold_smoke_1_preview`, which is equivalent to the source-safe baseline preview. For manual tuning, set `NOTEBOOK12_TEST_PROFILE = "custom"` and edit the overrides block.

The campaign smoke profiles still preserve guarded execution. `campaign_smoke_preview` prepares and previews a command only. `campaign_smoke_dry_run` requires both explicit execution permission and an advertised native dry-run argument before execution is allowed. `campaign_smoke_dry_run_allow_provisional` validates and explicitly allows the generated provisional config, but still blocks if no dry-run surface is advertised. The non-dry-run execution profile is intentionally named verbosely and should only be used when you deliberately want to run a tiny native campaign smoke.


In [ ]:

# -----------------------------------------------------------------------------
# Notebook 12 profile selector
# -----------------------------------------------------------------------------
# Change this one value for most smoke tests.
# You can also set NOTEBOOK12_TEST_PROFILE in the environment before running.
NOTEBOOK12_TEST_PROFILE = os.environ.get("NOTEBOOK12_TEST_PROFILE", "cold_smoke_5_command_shape_readiness").strip() or "cold_smoke_5_command_shape_readiness"

# Optional external inputs. These can be supplied through environment variables or
# edited here for a Colab/manual smoke run.
NOTEBOOK12_CAMPAIGN_ID_OVERRIDE = os.environ.get("NOTEBOOK12_CAMPAIGN_ID_FILTER", "").strip()
NOTEBOOK12_RUN_ID_OVERRIDE = os.environ.get("NOTEBOOK12_RUN_ID_FILTER", "").strip()
NOTEBOOK12_CAMPAIGN_SMOKE_ID_OVERRIDE = os.environ.get("NOTEBOOK12_CAMPAIGN_SMOKE_ID", "notebook12_campaign_smoke").strip()
NOTEBOOK12_CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE = os.environ.get("NOTEBOOK12_CAMPAIGN_SMOKE_CONFIG_PATH", "").strip()
NOTEBOOK12_ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_EXECUTION_OVERRIDE = os.environ.get("NOTEBOOK12_ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_EXECUTION", "").strip().lower()
NOTEBOOK12_VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_OVERRIDE = os.environ.get("NOTEBOOK12_VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG", "").strip().lower()
NOTEBOOK12_ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_OVERRIDE = os.environ.get("NOTEBOOK12_ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE", "").strip().lower()

BASE_RUNTIME_CONTROLS = {
    "NOTEBOOK12_MODE": "campaign_preview",
    "RUN_FINTECH_SESSION_INIT": True,
    "RUN_STRATLAKE_SESSION_INIT": True,
    "RUN_STRATLAKE_ARCHIVE_RESTORE": False,
    "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": False,
    "DISCOVER_CAMPAIGN_CONTEXT": False,
    "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": False,
    "RUN_CAMPAIGN_EVIDENCE_REVIEW": False,
    "RUN_CAMPAIGN_GOVERNANCE_REVIEW": False,
    "RUN_STRATLAKE_CAMPAIGN_REPORT": False,
    "RUN_STRATLAKE_ARCHIVE_CHECKPOINT": False,
    "RUN_NATIVE_CAMPAIGN_SMOKE": False,
    "ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION": False,
    "NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY": True,
    "REQUIRE_DRY_RUN_ARGUMENT_FOR_NATIVE_CAMPAIGN_SMOKE": True,
    "ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_EXECUTION": False,
    "NATIVE_CAMPAIGN_SMOKE_DRY_RUN_OPTION_CANDIDATES": ["--dry-run", "--dry_run", "--preview", "--check", "--validate-only"],
    "DISCOVER_NATIVE_CAMPAIGN_CONFIG_TEMPLATES": True,
    "WRITE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG": True,
    "ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION": False,
    "VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG": True,
    "RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY": True,
    "ALLOW_REFERENCE_ONLY_CAMPAIGN_PLAN": False,
    "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": False,
    "CAMPAIGN_ID_FILTER": None,
    "RUN_ID_FILTER": None,
    "REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW": False,
    "CAMPAIGN_DISCOVERY_MAX_FILES": 250,
    "CAMPAIGN_DISCOVERY_MAX_DEPTH": 8,
    "CAMPAIGN_DISCOVERY_INCLUDE_NOTEBOOK11_REVIEW_DIR": True,
    "CAMPAIGN_DISCOVERY_INCLUDE_NOTEBOOK12_REVIEW_DIR": True,
    "WRITE_NOTEBOOK12_ARTIFACTS": True,
    "CAMPAIGN_SMOKE_ID": NOTEBOOK12_CAMPAIGN_SMOKE_ID_OVERRIDE or "notebook12_campaign_smoke",
    "CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE": NOTEBOOK12_CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE,
    "CAMPAIGN_SMOKE_TEMPLATE_SEARCH_MAX_FILES": 50,
    "CAMPAIGN_SMOKE_TEMPLATE_SEARCH_MAX_DEPTH": 5,
}

NOTEBOOK12_TEST_PROFILES = {
    "cold_smoke_1_preview": {
        "description": "Baseline source-safe preview. No restore, campaign discovery, campaign run, evidence run, governance run, or checkpoint.",
        "overrides": {},
    },
    "cold_smoke_2_loose_restored_discovery": {
        "description": "Loose restored artifact discovery without archive restore or runtime evidence/governance execution.",
        "overrides": {
            "NOTEBOOK12_MODE": "restored_campaign_review",
            "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": True,
            "DISCOVER_CAMPAIGN_CONTEXT": True,
            "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
            "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
            "REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW": False,
        },
    },
    "cold_smoke_3_strict_negative_control": {
        "description": "Strict restored discovery with a synthetic missing campaign id. Validates no fabricated or mixed campaign context.",
        "overrides": {
            "NOTEBOOK12_MODE": "restored_campaign_review",
            "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": True,
            "DISCOVER_CAMPAIGN_CONTEXT": True,
            "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
            "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
            "REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW": True,
            "CAMPAIGN_ID_FILTER": "notebook12_missing_campaign_smoke",
        },
    },
    "cold_smoke_4_strict_missing_filter_guardrail": {
        "description": "Strict restored discovery with no campaign/run filter. Validates the missing-filter guardrail.",
        "overrides": {
            "NOTEBOOK12_MODE": "restored_campaign_review",
            "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": True,
            "DISCOVER_CAMPAIGN_CONTEXT": True,
            "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
            "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
            "REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW": True,
            "CAMPAIGN_ID_FILTER": None,
            "RUN_ID_FILTER": None,
        },
    },
    "cold_smoke_5_command_shape_readiness": {
        "description": "Command-shape regression posture. Detects native campaign/evidence/governance command shapes with execution disabled.",
        "overrides": {
            "NOTEBOOK12_MODE": "restored_campaign_review",
            "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": True,
            "DISCOVER_CAMPAIGN_CONTEXT": True,
            "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
            "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
            "REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW": False,
        },
    },
    "campaign_smoke_preview": {
        "description": "Prepare/discover a campaign smoke config and preview the native run command. Does not execute.",
        "overrides": {
            "NOTEBOOK12_MODE": "campaign_smoke_run",
            "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": True,
            "DISCOVER_CAMPAIGN_CONTEXT": True,
            "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
            "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
            "RUN_NATIVE_CAMPAIGN_SMOKE": True,
            "ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION": False,
            "NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY": True,
            "REQUIRE_DRY_RUN_ARGUMENT_FOR_NATIVE_CAMPAIGN_SMOKE": True,
        },
    },
    "campaign_smoke_dry_run": {
        "description": "Guarded native campaign smoke dry run. Execution only proceeds if explicitly allowed and the native CLI advertises dry-run support.",
        "overrides": {
            "NOTEBOOK12_MODE": "campaign_smoke_run",
            "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": True,
            "DISCOVER_CAMPAIGN_CONTEXT": True,
            "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
            "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
            "RUN_NATIVE_CAMPAIGN_SMOKE": True,
            "ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION": True,
            "NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY": True,
            "REQUIRE_DRY_RUN_ARGUMENT_FOR_NATIVE_CAMPAIGN_SMOKE": True,
        },
    },
    "campaign_smoke_dry_run_allow_provisional": {
        "description": "Guarded native campaign smoke dry run using the generated provisional config only when explicit provisional execution and validation both pass.",
        "overrides": {
            "NOTEBOOK12_MODE": "campaign_smoke_run",
            "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": True,
            "DISCOVER_CAMPAIGN_CONTEXT": True,
            "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
            "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
            "RUN_NATIVE_CAMPAIGN_SMOKE": True,
            "ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION": True,
            "NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY": True,
            "REQUIRE_DRY_RUN_ARGUMENT_FOR_NATIVE_CAMPAIGN_SMOKE": True,
            "ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION": True,
            "VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG": True,
        },
    },
    "campaign_smoke_execute_allow_provisional_no_dry_run": {
        "description": "Explicit non-dry-run native campaign smoke using the validated provisional config. This may run a tiny campaign and should only be used deliberately after preview/dry-run audits.",
        "overrides": {
            "NOTEBOOK12_MODE": "campaign_smoke_run",
            "DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT": True,
            "DISCOVER_CAMPAIGN_CONTEXT": True,
            "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
            "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
            "RUN_NATIVE_CAMPAIGN_SMOKE": True,
            "ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION": True,
            "NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY": False,
            "REQUIRE_DRY_RUN_ARGUMENT_FOR_NATIVE_CAMPAIGN_SMOKE": False,
            "ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_EXECUTION": True,
            "ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION": True,
            "VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG": True,
        },
    },
    "custom": {
        "description": "Manual profile. Edit CUSTOM_RUNTIME_CONTROL_OVERRIDES below.",
        "overrides": {},
    },
}

# Manual overrides are only applied when NOTEBOOK12_TEST_PROFILE == "custom".
CUSTOM_RUNTIME_CONTROL_OVERRIDES = {
    # Example:
    # "NOTEBOOK12_MODE": "restored_campaign_review",
    # "RUN_CAMPAIGN_ARTIFACT_DISCOVERY": True,
    # "DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS": True,
    # "CAMPAIGN_ID_FILTER": "your_campaign_id_here",
    # "ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION": True,
    # "VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG": True,
    # "ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_EXECUTION": False,
}

if NOTEBOOK12_TEST_PROFILE not in NOTEBOOK12_TEST_PROFILES:
    print(f"Unknown NOTEBOOK12_TEST_PROFILE={NOTEBOOK12_TEST_PROFILE!r}; falling back to cold_smoke_1_preview.")
    NOTEBOOK12_TEST_PROFILE = "cold_smoke_1_preview"

runtime_controls = dict(BASE_RUNTIME_CONTROLS)
runtime_controls.update(NOTEBOOK12_TEST_PROFILES[NOTEBOOK12_TEST_PROFILE]["overrides"])
if NOTEBOOK12_TEST_PROFILE == "custom":
    runtime_controls.update(CUSTOM_RUNTIME_CONTROL_OVERRIDES)

# Environment variables take precedence for campaign/run filters when provided.
if NOTEBOOK12_CAMPAIGN_ID_OVERRIDE:
    runtime_controls["CAMPAIGN_ID_FILTER"] = NOTEBOOK12_CAMPAIGN_ID_OVERRIDE
if NOTEBOOK12_RUN_ID_OVERRIDE:
    runtime_controls["RUN_ID_FILTER"] = NOTEBOOK12_RUN_ID_OVERRIDE
if NOTEBOOK12_CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE:
    runtime_controls["CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE"] = NOTEBOOK12_CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE
if NOTEBOOK12_ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_EXECUTION_OVERRIDE in {"1", "true", "yes", "y"}:
    runtime_controls["ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION"] = True
elif NOTEBOOK12_ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_EXECUTION_OVERRIDE in {"0", "false", "no", "n"}:
    runtime_controls["ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION"] = False
if NOTEBOOK12_VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_OVERRIDE in {"1", "true", "yes", "y"}:
    runtime_controls["VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG"] = True
elif NOTEBOOK12_VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_OVERRIDE in {"0", "false", "no", "n"}:
    runtime_controls["VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG"] = False
if NOTEBOOK12_ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_OVERRIDE in {"1", "true", "yes", "y"}:
    runtime_controls["ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_EXECUTION"] = True
elif NOTEBOOK12_ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_OVERRIDE in {"0", "false", "no", "n"}:
    runtime_controls["ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_EXECUTION"] = False

# Export runtime controls as ordinary notebook globals so downstream cells remain simple.
globals().update(runtime_controls)

SOURCE_SAFE_DEFAULTS_ACTIVE = (
    NOTEBOOK12_MODE == "campaign_preview"
    and not RUN_STRATLAKE_ARCHIVE_RESTORE
    and not DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT
    and not DISCOVER_CAMPAIGN_CONTEXT
    and not RUN_CAMPAIGN_ARTIFACT_DISCOVERY
    and not RUN_CAMPAIGN_EVIDENCE_REVIEW
    and not RUN_CAMPAIGN_GOVERNANCE_REVIEW
    and not RUN_STRATLAKE_CAMPAIGN_REPORT
    and not RUN_STRATLAKE_ARCHIVE_CHECKPOINT
    and not RUN_NATIVE_CAMPAIGN_SMOKE
)

RESTORED_REVIEW_ARTIFACT_DISCOVERY_REQUESTED = (
    NOTEBOOK12_MODE == "restored_campaign_review"
    or DISCOVER_CAMPAIGN_CONTEXT
    or RUN_CAMPAIGN_ARTIFACT_DISCOVERY
)

profile_catalog_df = pd.DataFrame([
    {
        "profile": profile_name,
        "description": profile_spec["description"],
        "mode": dict(BASE_RUNTIME_CONTROLS, **profile_spec["overrides"]).get("NOTEBOOK12_MODE"),
        "runs_native_campaign_smoke": dict(BASE_RUNTIME_CONTROLS, **profile_spec["overrides"]).get("RUN_NATIVE_CAMPAIGN_SMOKE"),
        "allows_native_campaign_smoke_execution": dict(BASE_RUNTIME_CONTROLS, **profile_spec["overrides"]).get("ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION"),
        "allows_provisional_campaign_smoke_config_execution": dict(BASE_RUNTIME_CONTROLS, **profile_spec["overrides"]).get("ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION"),
        "validates_provisional_campaign_smoke_config": dict(BASE_RUNTIME_CONTROLS, **profile_spec["overrides"]).get("VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG"),
        "runs_campaign_artifact_discovery": dict(BASE_RUNTIME_CONTROLS, **profile_spec["overrides"]).get("RUN_CAMPAIGN_ARTIFACT_DISCOVERY"),
        "strict_filter_required": dict(BASE_RUNTIME_CONTROLS, **profile_spec["overrides"]).get("REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW"),
    }
    for profile_name, profile_spec in NOTEBOOK12_TEST_PROFILES.items()
])

runtime_control_summary = {
    "notebook12_test_profile": NOTEBOOK12_TEST_PROFILE,
    "notebook12_test_profile_description": NOTEBOOK12_TEST_PROFILES[NOTEBOOK12_TEST_PROFILE]["description"],
    "notebook12_mode": NOTEBOOK12_MODE,
    "source_safe_defaults_active": SOURCE_SAFE_DEFAULTS_ACTIVE,
    "restored_review_artifact_discovery_requested": RESTORED_REVIEW_ARTIFACT_DISCOVERY_REQUESTED,
    "run_id_strict_platform_artifact_discovery": RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY,
    "allow_reference_only_campaign_plan": ALLOW_REFERENCE_ONLY_CAMPAIGN_PLAN,
    "discover_existing_campaign_artifacts": DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS,
    "campaign_id_filter": CAMPAIGN_ID_FILTER,
    "run_id_filter": RUN_ID_FILTER,
    "require_campaign_or_run_filter_for_restored_review": REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW,
    "campaign_discovery_max_files": CAMPAIGN_DISCOVERY_MAX_FILES,
    "campaign_discovery_max_depth": CAMPAIGN_DISCOVERY_MAX_DEPTH,
    "run_native_campaign_smoke": RUN_NATIVE_CAMPAIGN_SMOKE,
    "allow_native_campaign_smoke_execution": ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION,
    "native_campaign_smoke_dry_run_only": NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY,
    "require_dry_run_argument_for_native_campaign_smoke": REQUIRE_DRY_RUN_ARGUMENT_FOR_NATIVE_CAMPAIGN_SMOKE,
    "campaign_smoke_id": CAMPAIGN_SMOKE_ID,
    "campaign_smoke_config_path_override": CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE or None,
    "created_at_utc": utc_now_iso(),
}
print_json(runtime_control_summary)

try:
    display(profile_catalog_df)
except Exception:
    print(profile_catalog_df.to_string(index=False))



### Manual restored campaign review recipe

Uncomment only after reviewing the archive target and campaign identifiers. This mode restores or attaches existing artifacts and reviews them; it still does **not** execute a new campaign by default.

```python
NOTEBOOK12_MODE = "restored_campaign_review"
RUN_STRATLAKE_ARCHIVE_RESTORE = True
DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT = True
DISCOVER_CAMPAIGN_CONTEXT = True
RUN_CAMPAIGN_ARTIFACT_DISCOVERY = True
RUN_CAMPAIGN_EVIDENCE_REVIEW = True
RUN_CAMPAIGN_GOVERNANCE_REVIEW = True
DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS = True

# Strongly preferred when known:
CAMPAIGN_ID_FILTER = "REPLACE_WITH_CAMPAIGN_ID"
RUN_ID_FILTER = None
```

Do not enable runtime campaign execution by default. A restored review should discover existing platform artifacts, classify evidence completeness, and report caveats without approving or promoting strategies.


## 4. Workspace, Google Drive mount, and path setup

Keep `/content` as the active runtime workspace in Colab. Treat Google Drive as archive/session persistence, not as the active application workspace.

This cell follows the prior Notebook 11 procedure by mounting Google Drive in live Colab via `drive.mount("/content/drive")`. Outside Colab it skips the mount and continues source-safely.

No private Drive path is hardcoded.


In [ ]:
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()
DRIVE_FOLDER_NAME = "TEST1"

print("IN_COLAB:", IN_COLAB)
print("Python executable:", sys.executable)
print("Current working directory:", Path.cwd().as_posix())

if IN_COLAB and drive is not None:
    # Match the established Notebook 11 Colab procedure:
    # mount Google Drive during workspace setup so archive/session restore
    # and checkpoint cells can resolve /content/drive/MyDrive paths.
    drive.mount("/content/drive")
else:
    print("Not running in Colab; skipping Google Drive mount.")

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME if IN_COLAB else None

FINTECH_ROOT = WORKSPACE_ROOT / "fintech-market-ingestion-demo"
STRATLAKE_ROOT = WORKSPACE_ROOT / "stratlake-trade-engine-demo"

NOTEBOOK12_REVIEW_DIR = STRATLAKE_ROOT / "artifacts" / "notebook_12_campaign_evidence_gap_promotion_readiness"
NOTEBOOK11_REVIEW_DIR = STRATLAKE_ROOT / "artifacts" / "notebook_11_expanded_promotion_evidence_review"
CAMPAIGN_ARTIFACT_ROOT = STRATLAKE_ROOT / "artifacts"
STRATLAKE_CONFIG_DIR = STRATLAKE_ROOT / "configs"
NATIVE_CAMPAIGN_SMOKE_CONFIG_DIR = STRATLAKE_CONFIG_DIR / "notebook12"

print("WORKSPACE_ROOT:", WORKSPACE_ROOT.as_posix())
print("FINTECH_ROOT:", FINTECH_ROOT.as_posix())
print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("DRIVE_ROOT:", DRIVE_ROOT.as_posix() if DRIVE_ROOT else None)
print("NOTEBOOK12_REVIEW_DIR:", NOTEBOOK12_REVIEW_DIR.as_posix())
print("NATIVE_CAMPAIGN_SMOKE_CONFIG_DIR:", NATIVE_CAMPAIGN_SMOKE_CONFIG_DIR.as_posix())


## Derived StratLake session archive root discovery

Derive the session archive root from the mounted Google Drive workspace first. Optionally search a bounded set of likely Drive locations for an existing `session_archives` directory.

This avoids hardcoded placeholder restore/checkpoint paths while keeping Google Drive as archive/session persistence rather than the active application workspace.

In [ ]:
SEARCH_DRIVE_FOR_SESSION_ARCHIVE_ROOT = True
SESSION_ARCHIVE_DIR_NAME = "session_archives"
SESSION_ARCHIVE_SEARCH_MAX_DEPTH = 5
SESSION_ARCHIVE_SEARCH_MAX_RESULTS = 20


def _path_depth_from_root(path: Path, root: Path) -> int:
    try:
        return len(path.relative_to(root).parts)
    except ValueError:
        return 999


def discover_session_archive_roots(
    search_roots: list[Path],
    dirname: str = SESSION_ARCHIVE_DIR_NAME,
    max_depth: int = SESSION_ARCHIVE_SEARCH_MAX_DEPTH,
    max_results: int = SESSION_ARCHIVE_SEARCH_MAX_RESULTS,
) -> list[Path]:
    """Return likely StratLake session archive roots using a bounded Drive search."""
    discovered: list[Path] = []
    seen: set[str] = set()

    for root in search_roots:
        if root is None:
            continue
        root = Path(root)
        if not root.exists():
            continue

        # Check the root itself and known direct children cheaply first.
        direct_candidates = [
            root,
            root / dirname,
            root / "stratlake-colab" / dirname,
            root / "session_archives" / "stratlake",
        ]
        for candidate in direct_candidates:
            if candidate.name == dirname and candidate.exists() and candidate.is_dir():
                key = candidate.resolve().as_posix()
                if key not in seen:
                    discovered.append(candidate)
                    seen.add(key)

        if not SEARCH_DRIVE_FOR_SESSION_ARCHIVE_ROOT:
            continue

        # Bounded walk avoids scanning the full Drive indefinitely.
        for current_root, dirnames, _filenames in os.walk(root):
            current_path = Path(current_root)
            if _path_depth_from_root(current_path, root) >= max_depth:
                dirnames[:] = []
                continue

            matches = [name for name in dirnames if name == dirname]
            for name in matches:
                candidate = current_path / name
                key = candidate.resolve().as_posix()
                if key not in seen:
                    discovered.append(candidate)
                    seen.add(key)
                    if len(discovered) >= max_results:
                        return discovered
    return discovered


DEFAULT_STRATLAKE_SESSION_ARCHIVE_ROOT = (
    DRIVE_ROOT / "stratlake-colab" / "session_archives"
    if DRIVE_ROOT is not None
    else WORKSPACE_ROOT / "stratlake-colab" / "session_archives"
)

SESSION_ARCHIVE_SEARCH_ROOTS = [
    DRIVE_ROOT / "stratlake-colab" if DRIVE_ROOT is not None else None,
    DRIVE_ROOT if DRIVE_ROOT is not None else None,
    Path("/content/drive/MyDrive") if IN_COLAB else None,
]
SESSION_ARCHIVE_SEARCH_ROOTS = [root for root in SESSION_ARCHIVE_SEARCH_ROOTS if root is not None]

_discovered_session_archive_roots = discover_session_archive_roots(SESSION_ARCHIVE_SEARCH_ROOTS)

if _discovered_session_archive_roots:
    STRATLAKE_SESSION_ARCHIVE_ROOT = _discovered_session_archive_roots[0]
    session_archive_root_discovery_status = "session_archive_root_discovered"
else:
    STRATLAKE_SESSION_ARCHIVE_ROOT = DEFAULT_STRATLAKE_SESSION_ARCHIVE_ROOT
    session_archive_root_discovery_status = "session_archive_root_default_selected"

SESSION_ARCHIVE_ROOT_DISCOVERY = {
    "status": session_archive_root_discovery_status,
    "selected_root": STRATLAKE_SESSION_ARCHIVE_ROOT.as_posix(),
    "default_root": DEFAULT_STRATLAKE_SESSION_ARCHIVE_ROOT.as_posix(),
    "search_roots": [root.as_posix() for root in SESSION_ARCHIVE_SEARCH_ROOTS],
    "discovered_roots": [root.as_posix() for root in _discovered_session_archive_roots],
    "search_enabled": SEARCH_DRIVE_FOR_SESSION_ARCHIVE_ROOT,
    "max_depth": SESSION_ARCHIVE_SEARCH_MAX_DEPTH,
    "max_results": SESSION_ARCHIVE_SEARCH_MAX_RESULTS,
}

print(json.dumps(SESSION_ARCHIVE_ROOT_DISCOVERY, indent=2))

## 5. CLI and import availability

Campaign-related commands are discovered defensively. Absence is recorded as a caveat, not treated as a notebook failure.

Expected existing or possible surfaces:

- `stratlake-run-research-campaign`
- `stratlake-build-campaign-report`
- `stratlake-build-evidence-review`
- `stratlake-run-promotion-governance-report`
- `stratlake-session-archive-restore-bootstrap`
- `stratlake-session-archive-bootstrap`


In [ ]:
EXPECTED_STRATLAKE_COMMANDS = [
    "stratlake-run-research-campaign",
    "stratlake-build-campaign-report",
    "stratlake-build-evidence-review",
    "stratlake-run-promotion-governance-report",
    "stratlake-session-archive-restore-bootstrap",
    "stratlake-session-archive-bootstrap",
]

EXPECTED_IMPORT_SURFACES = [
    "src.execution",
    "src.execution.run_research_campaign",
    "src.execution.run_strategy",
    "src.execution.compare_strategies",
    "src.research.promotion",
]


def command_available(command: str) -> bool:
    return shutil.which(command) is not None


def command_help_preview(command: str, timeout_seconds: int = 20) -> dict[str, Any]:
    if not command_available(command):
        return {
            "command": command,
            "available": False,
            "help_checked": False,
            "status": "cli_unavailable",
        }
    try:
        result = subprocess.run(
            [command, "--help"],
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            check=False,
        )
        return {
            "command": command,
            "available": True,
            "help_checked": True,
            "returncode": result.returncode,
            "stdout_head": result.stdout[:1200],
            "stderr_head": result.stderr[:1200],
            "status": "cli_help_loaded" if result.returncode == 0 else "cli_help_returned_nonzero",
        }
    except Exception as exc:
        return {
            "command": command,
            "available": True,
            "help_checked": False,
            "error": repr(exc),
            "status": "cli_help_error",
        }


def import_surface_available(module_name: str) -> dict[str, Any]:
    try:
        importlib.import_module(module_name)
        return {"module": module_name, "available": True, "status": "import_surface_loaded"}
    except Exception as exc:
        return {
            "module": module_name,
            "available": False,
            "status": "import_surface_unavailable",
            "error": repr(exc)[:500],
        }


cli_surface_rows = [command_help_preview(cmd) for cmd in EXPECTED_STRATLAKE_COMMANDS]
import_surface_rows = [import_surface_available(name) for name in EXPECTED_IMPORT_SURFACES]

cli_surface_df = pd.DataFrame(cli_surface_rows)
import_surface_df = pd.DataFrame(import_surface_rows)

print("CLI surfaces")
show_df(cli_surface_df)
print("\nImport surfaces")
show_df(import_surface_df)


## 6. Initialize or attach Fintech project/session

This keeps the established Notebook 10/11 initialization pattern. Review-only campaign artifact inspection should not require Alpaca credentials.

If the native CLI is unavailable, record the gap and continue.


In [ ]:
FINTECH_INIT_COMMAND = [
    "fintech-session-init",
    "--root", str(FINTECH_ROOT),
]

fintech_init_result = {
    "command": " ".join(shlex.quote(part) for part in FINTECH_INIT_COMMAND),
    "run_requested": RUN_FINTECH_SESSION_INIT,
    "status": "not_run",
}

if RUN_FINTECH_SESSION_INIT:
    if command_available(FINTECH_INIT_COMMAND[0]):
        try:
            result = subprocess.run(
                FINTECH_INIT_COMMAND,
                capture_output=True,
                text=True,
                timeout=60,
                check=False,
            )
            fintech_init_result.update({
                "returncode": result.returncode,
                "stdout_head": result.stdout[:1200],
                "stderr_head": result.stderr[:1200],
                "status": "fintech_session_init_completed" if result.returncode == 0 else "fintech_session_init_returned_nonzero",
            })
        except Exception as exc:
            fintech_init_result.update({"status": "fintech_session_init_error", "error": repr(exc)})
    else:
        fintech_init_result.update({"status": "fintech_session_init_cli_unavailable"})
else:
    fintech_init_result.update({"status": "fintech_session_init_skipped"})

print_json(fintech_init_result)


## 7. Initialize or attach StratLake session

This cell initializes or attaches to the StratLake workspace when the native session command is available. It does not execute a research campaign.


In [ ]:
STRATLAKE_INIT_CANDIDATES = [
    "stratlake-session-init",
    "stratlake-session-bootstrap",
]

stratlake_init_result = {
    "run_requested": RUN_STRATLAKE_SESSION_INIT,
    "candidate_commands": STRATLAKE_INIT_CANDIDATES,
    "status": "not_run",
}

if RUN_STRATLAKE_SESSION_INIT:
    init_command = next((cmd for cmd in STRATLAKE_INIT_CANDIDATES if command_available(cmd)), None)
    if init_command is None:
        ensure_dir(STRATLAKE_ROOT)
        stratlake_init_result.update({
            "status": "stratlake_session_init_cli_unavailable_directory_prepared_only",
            "stratlake_root": str(STRATLAKE_ROOT),
        })
    else:
        command = [init_command, "--root", str(STRATLAKE_ROOT)]
        try:
            result = subprocess.run(
                command,
                capture_output=True,
                text=True,
                timeout=60,
                check=False,
            )
            stratlake_init_result.update({
                "command": " ".join(shlex.quote(part) for part in command),
                "returncode": result.returncode,
                "stdout_head": result.stdout[:1200],
                "stderr_head": result.stderr[:1200],
                "status": "stratlake_session_init_completed" if result.returncode == 0 else "stratlake_session_init_returned_nonzero",
            })
        except Exception as exc:
            stratlake_init_result.update({"status": "stratlake_session_init_error", "error": repr(exc)})
else:
    stratlake_init_result.update({"status": "stratlake_session_init_skipped"})

print_json(stratlake_init_result)


## 8. Guarded native campaign execution smoke path

This section is the direct bridge from campaign readiness into campaign execution. It remains off by default.

The notebook first tries to discover an existing native StratLake campaign config template. If no native template is found, it can write a provisional reference config for human editing, but it will not execute that provisional config unless explicitly allowed.

Execution requires all of the following:

```python
NOTEBOOK12_MODE = "campaign_smoke_run"
RUN_NATIVE_CAMPAIGN_SMOKE = True
ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION = True
```

By default the smoke path also requires an advertised dry-run argument. If the installed CLI does not advertise dry-run support, the notebook blocks execution rather than accidentally running a live campaign.

In [ ]:

def path_within_depth(path: Path, root: Path, max_depth: int) -> bool:
    try:
        rel = path.relative_to(root)
    except Exception:
        return False
    return len(rel.parts) <= max_depth


def smoke_config_source_for_path(path: Path) -> dict[str, Any]:
    """Return an audit-friendly source classification for a campaign smoke config.

    Notebook 12 writes a generated smoke config under NATIVE_CAMPAIGN_SMOKE_CONFIG_DIR.
    When that file persists across cells/runs, discovery can find it later. It should
    remain useful for preview/provisional execution checks, but it should not be
    labeled as a true native StratLake template.
    """
    generated_dir = globals().get("NATIVE_CAMPAIGN_SMOKE_CONFIG_DIR")
    generated_id = str(globals().get("CAMPAIGN_SMOKE_ID", "notebook12_campaign_smoke"))
    try:
        is_generated_dir = bool(generated_dir and path.resolve().is_relative_to(Path(generated_dir).resolve()))
    except Exception:
        try:
            is_generated_dir = bool(generated_dir and path.resolve().as_posix().startswith(Path(generated_dir).resolve().as_posix()))
        except Exception:
            is_generated_dir = False
    is_generated_name = path.stem == generated_id or path.name in {f"{generated_id}.yml", f"{generated_id}.yaml", f"{generated_id}.json"}
    is_notebook_generated = bool(is_generated_dir or is_generated_name)
    return {
        "config_source": "notebook12_generated_smoke_config" if is_notebook_generated else "native_campaign_template",
        "config_is_native_template": not is_notebook_generated,
        "config_is_notebook_generated": is_notebook_generated,
        "status": "candidate_notebook12_generated_smoke_config" if is_notebook_generated else "candidate_native_campaign_config_template",
    }


def discover_campaign_config_templates(root: Path, max_files: int = 50, max_depth: int = 5) -> pd.DataFrame:
    search_roots = [
        root / "configs",
        root / "config",
        root / "examples",
        root / "docs",
        root / "tests",
    ]
    rows: list[dict[str, Any]] = []
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for pattern in ("*.yml", "*.yaml", "*.json"):
            for path in search_root.rglob(pattern):
                if len(rows) >= max_files:
                    break
                if not path.is_file() or not path_within_depth(path, search_root, max_depth):
                    continue
                lowered = path.as_posix().lower()
                if "campaign" not in lowered and "research" not in lowered:
                    continue
                source_info = smoke_config_source_for_path(path)
                rows.append({
                    "path": path.as_posix(),
                    "name": path.name,
                    "search_root": search_root.as_posix(),
                    "suffix": path.suffix,
                    "size_bytes": path.stat().st_size,
                    "status": source_info["status"],
                    "config_source": source_info["config_source"],
                    "config_is_native_template": source_info["config_is_native_template"],
                    "config_is_notebook_generated": source_info["config_is_notebook_generated"],
                })
            if len(rows) >= max_files:
                break
    columns = [
        "path", "name", "search_root", "suffix", "size_bytes", "status",
        "config_source", "config_is_native_template", "config_is_notebook_generated",
    ]
    return pd.DataFrame(rows, columns=columns)


def provisional_campaign_smoke_config_text() -> str:
    """Return a conservative reference config for human review.

    This is intentionally marked provisional because the exact StratLake campaign
    schema should remain owned by the platform. Prefer a native template or an
    explicit NOTEBOOK12_CAMPAIGN_SMOKE_CONFIG_PATH for execution.
    """
    return f"""# Notebook 12 provisional campaign smoke config.
# Source-safe by default. Validate this file against the installed StratLake
# campaign CLI/schema before enabling execution.
#
# Preferred: replace this file with a native platform template or pass
# NOTEBOOK12_CAMPAIGN_SMOKE_CONFIG_PATH to a known-good campaign config.

campaign_id: {CAMPAIGN_SMOKE_ID}
description: Notebook 12 guarded native campaign smoke.
artifacts_root: {CAMPAIGN_ARTIFACT_ROOT.as_posix()}
review_output_dir: {NOTEBOOK12_REVIEW_DIR.as_posix()}

universe:
  symbols:
    - SPY
    - QQQ
  data_frequency: daily

windows:
  - window_name: smoke_window
    start: '2024-01-02'
    end: '2024-03-28'

strategies:
  - strategy_id: buy_and_hold_v1
  - strategy_id: sma_crossover_v1

execution:
  source_safe_smoke: true
  max_runs: 2
  require_existing_feature_data: true
  write_artifacts: true

governance:
  promotion_claims_allowed: false
  require_human_review: true
"""


def validate_provisional_campaign_smoke_config(path: Path) -> dict[str, Any]:
    """Conservative local validation for the generated Notebook 12 smoke config.

    This does not claim platform-schema validity. It only verifies that the
    generated reference config contains the minimum fields Notebook 12 expects
    before allowing a guarded dry-run attempt.
    """
    required_top_level_keys = [
        "campaign_id",
        "description",
        "artifacts_root",
        "review_output_dir",
        "universe",
        "windows",
        "strategies",
        "execution",
        "governance",
    ]
    required_markers = [
        "symbols:",
        "window_name:",
        "start:",
        "end:",
        "strategy_id:",
        "source_safe_smoke:",
        "promotion_claims_allowed: false",
        "require_human_review: true",
    ]
    result: dict[str, Any] = {
        "path": path.as_posix(),
        "validator": "notebook12_conservative_text_scan",
        "valid": False,
        "errors": [],
        "warnings": [],
        "required_top_level_keys_present": [],
        "required_top_level_keys_missing": [],
        "required_markers_present": [],
        "required_markers_missing": [],
    }
    if not path.exists():
        result["errors"].append("provisional_config_missing")
        return result
    text = path.read_text(encoding="utf-8")
    if not text.strip():
        result["errors"].append("provisional_config_empty")
        return result

    for key in required_top_level_keys:
        marker = f"{key}:"
        if re.search(rf"(?m)^\s*{re.escape(marker)}", text):
            result["required_top_level_keys_present"].append(key)
        else:
            result["required_top_level_keys_missing"].append(key)

    for marker in required_markers:
        if marker in text:
            result["required_markers_present"].append(marker)
        else:
            result["required_markers_missing"].append(marker)

    if str(CAMPAIGN_SMOKE_ID) not in text:
        result["warnings"].append("campaign_smoke_id_not_found_in_config_text")
    if CAMPAIGN_ARTIFACT_ROOT.as_posix() not in text:
        result["warnings"].append("campaign_artifact_root_not_found_in_config_text")
    if NOTEBOOK12_REVIEW_DIR.as_posix() not in text:
        result["warnings"].append("notebook12_review_dir_not_found_in_config_text")

    if result["required_top_level_keys_missing"]:
        result["errors"].append("missing_required_top_level_keys")
    if result["required_markers_missing"]:
        result["errors"].append("missing_required_markers")

    result["valid"] = not result["errors"]
    return result


def help_text_for_command(command: str) -> str:
    preview = command_help_preview(command)
    return "\n".join([
        str(preview.get("stdout_head", "") or ""),
        str(preview.get("stderr_head", "") or ""),
    ])


def extract_advertised_options_from_help(help_text: str) -> list[str]:
    """Extract long option-like tokens from CLI help text for audit diagnostics."""
    if not help_text:
        return []
    return sorted(set(re.findall(r"(?<![\w-])--[A-Za-z0-9][A-Za-z0-9_-]*", help_text)))


def command_supports_option(command: str, option: str) -> bool:
    return option in help_text_for_command(command)


def dry_run_support_diagnostics(command: str, candidates: list[str]) -> dict[str, Any]:
    help_text = help_text_for_command(command)
    advertised = extract_advertised_options_from_help(help_text)
    matched = [option for option in candidates if option in advertised or option in help_text]
    return {
        "command": command,
        "candidate_options": candidates,
        "advertised_options": advertised,
        "matched_options": matched,
        "selected_option": matched[0] if matched else None,
        "supports_dry_run": bool(matched),
        "help_text_captured": bool(help_text.strip()),
        "status": "dry_run_option_advertised" if matched else "dry_run_option_not_advertised",
    }


native_campaign_template_df = pd.DataFrame()
if DISCOVER_NATIVE_CAMPAIGN_CONFIG_TEMPLATES:
    native_campaign_template_df = discover_campaign_config_templates(
        STRATLAKE_ROOT,
        max_files=CAMPAIGN_SMOKE_TEMPLATE_SEARCH_MAX_FILES,
        max_depth=CAMPAIGN_SMOKE_TEMPLATE_SEARCH_MAX_DEPTH,
    )

ensure_dir(NOTEBOOK12_REVIEW_DIR)
ensure_dir(NATIVE_CAMPAIGN_SMOKE_CONFIG_DIR)
provisional_campaign_smoke_config_path = NATIVE_CAMPAIGN_SMOKE_CONFIG_DIR / f"{CAMPAIGN_SMOKE_ID}.yml"
provisional_campaign_smoke_validation: dict[str, Any] | None = None
selected_campaign_smoke_config_source: str | None = None
selected_campaign_smoke_config_is_native_template = False
selected_campaign_smoke_config_is_notebook_generated = False
selected_campaign_smoke_config_validation_status = "validation_not_applicable"

if WRITE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG:
    provisional_campaign_smoke_config_path.write_text(provisional_campaign_smoke_config_text(), encoding="utf-8")

# Separate previewability from executability.
# A preview may use a provisional reference config so the user can inspect the
# exact native command. Execution still requires an explicitly executable config.
preview_campaign_smoke_config_path: Path | None = None
executable_campaign_smoke_config_path: Path | None = None
campaign_smoke_config_status = "no_campaign_smoke_config_available"

native_campaign_config_template_df = native_campaign_template_df.loc[
    native_campaign_template_df.get("config_is_native_template", pd.Series(dtype=bool)).fillna(False)
].copy() if not native_campaign_template_df.empty else pd.DataFrame()
notebook_generated_smoke_config_df = native_campaign_template_df.loc[
    native_campaign_template_df.get("config_is_notebook_generated", pd.Series(dtype=bool)).fillna(False)
].copy() if not native_campaign_template_df.empty else pd.DataFrame()

if CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE:
    override_path = Path(CAMPAIGN_SMOKE_CONFIG_PATH_OVERRIDE).expanduser()
    if override_path.exists():
        source_info = smoke_config_source_for_path(override_path)
        preview_campaign_smoke_config_path = override_path
        executable_campaign_smoke_config_path = override_path
        selected_campaign_smoke_config_source = "user_supplied_campaign_config"
        selected_campaign_smoke_config_is_native_template = False
        selected_campaign_smoke_config_is_notebook_generated = bool(source_info.get("config_is_notebook_generated", False))
        selected_campaign_smoke_config_validation_status = "user_supplied_config_notebook_validation_skipped"
        campaign_smoke_config_status = "user_supplied_campaign_config_selected"
    else:
        campaign_smoke_config_status = "user_supplied_campaign_config_missing"
elif not native_campaign_config_template_df.empty:
    template_path = Path(str(native_campaign_config_template_df.iloc[0]["path"]))
    preview_campaign_smoke_config_path = template_path
    executable_campaign_smoke_config_path = template_path
    selected_campaign_smoke_config_source = "native_campaign_template"
    selected_campaign_smoke_config_is_native_template = True
    selected_campaign_smoke_config_is_notebook_generated = False
    selected_campaign_smoke_config_validation_status = "native_template_notebook_validation_skipped"
    campaign_smoke_config_status = "native_campaign_template_selected"
elif provisional_campaign_smoke_config_path.exists():
    preview_campaign_smoke_config_path = provisional_campaign_smoke_config_path
    selected_campaign_smoke_config_source = "notebook12_generated_smoke_config"
    selected_campaign_smoke_config_is_native_template = False
    selected_campaign_smoke_config_is_notebook_generated = True
    provisional_campaign_smoke_validation = validate_provisional_campaign_smoke_config(provisional_campaign_smoke_config_path)
    selected_campaign_smoke_config_validation_status = (
        "validated_existing_notebook12_generated_smoke_config"
        if provisional_campaign_smoke_validation.get("valid", False)
        else "existing_notebook12_generated_smoke_config_validation_failed"
    )
    if ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION:
        if (not VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG) or provisional_campaign_smoke_validation.get("valid", False):
            executable_campaign_smoke_config_path = provisional_campaign_smoke_config_path
            campaign_smoke_config_status = "validated_notebook12_generated_smoke_config_selected_for_execution" if VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG else "unvalidated_notebook12_generated_smoke_config_selected_for_execution"
            if not VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG:
                selected_campaign_smoke_config_validation_status = "validation_disabled_for_existing_notebook12_generated_smoke_config"
        else:
            campaign_smoke_config_status = "notebook12_generated_smoke_config_validation_failed"
    else:
        campaign_smoke_config_status = "notebook12_generated_smoke_config_selected_reference_only"

campaign_smoke_command = None
native_campaign_cli_available = command_available("stratlake-run-research-campaign")
native_campaign_smoke_dry_run_diagnostics = dry_run_support_diagnostics(
    "stratlake-run-research-campaign",
    list(NATIVE_CAMPAIGN_SMOKE_DRY_RUN_OPTION_CANDIDATES),
) if native_campaign_cli_available else {
    "command": "stratlake-run-research-campaign",
    "candidate_options": list(NATIVE_CAMPAIGN_SMOKE_DRY_RUN_OPTION_CANDIDATES),
    "advertised_options": [],
    "matched_options": [],
    "selected_option": None,
    "supports_dry_run": False,
    "help_text_captured": False,
    "status": "cli_unavailable",
}
selected_dry_run_option = native_campaign_smoke_dry_run_diagnostics.get("selected_option")

if native_campaign_cli_available and preview_campaign_smoke_config_path is not None:
    campaign_smoke_command = [
        "stratlake-run-research-campaign",
        "--config",
        str(preview_campaign_smoke_config_path),
    ]
    if NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY and selected_dry_run_option:
        campaign_smoke_command.append(str(selected_dry_run_option))

can_build_native_campaign_smoke_preview = bool(
    RUN_NATIVE_CAMPAIGN_SMOKE
    and native_campaign_cli_available
    and preview_campaign_smoke_config_path is not None
    and campaign_smoke_command is not None
)

can_execute_native_campaign_smoke = bool(
    can_build_native_campaign_smoke_preview
    and ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION
    and executable_campaign_smoke_config_path is not None
    and (NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY or ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_EXECUTION)
)

native_campaign_smoke_result = {
    "run_requested": RUN_NATIVE_CAMPAIGN_SMOKE,
    "execution_allowed": ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION,
    "dry_run_only": NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY,
    "require_dry_run_argument": REQUIRE_DRY_RUN_ARGUMENT_FOR_NATIVE_CAMPAIGN_SMOKE,
    "allow_non_dry_run_execution": ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_EXECUTION,
    "dry_run_diagnostics": native_campaign_smoke_dry_run_diagnostics,
    "dry_run_support_status": native_campaign_smoke_dry_run_diagnostics.get("status"),
    "dry_run_supported": bool(native_campaign_smoke_dry_run_diagnostics.get("supports_dry_run")),
    "dry_run_selected_option": native_campaign_smoke_dry_run_diagnostics.get("selected_option"),
    "provisional_config_execution_allowed": ALLOW_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG_FOR_EXECUTION,
    "provisional_config_validation_required": VALIDATE_PROVISIONAL_CAMPAIGN_SMOKE_CONFIG,
    "provisional_config_validation": provisional_campaign_smoke_validation,
    "campaign_smoke_id": CAMPAIGN_SMOKE_ID,
    "template_candidates": int(len(native_campaign_template_df)),
    "provisional_config_path": provisional_campaign_smoke_config_path.as_posix(),
    "preview_config_path": preview_campaign_smoke_config_path.as_posix() if preview_campaign_smoke_config_path else None,
    "selected_config_path": preview_campaign_smoke_config_path.as_posix() if preview_campaign_smoke_config_path else None,
    "executable_config_path": executable_campaign_smoke_config_path.as_posix() if executable_campaign_smoke_config_path else None,
    "config_status": campaign_smoke_config_status,
    "config_source": selected_campaign_smoke_config_source,
    "config_is_native_template": bool(selected_campaign_smoke_config_is_native_template),
    "config_is_notebook_generated": bool(selected_campaign_smoke_config_is_notebook_generated),
    "config_validation_status": selected_campaign_smoke_config_validation_status,
    "native_template_candidate_count": int(len(native_campaign_config_template_df)) if "native_campaign_config_template_df" in globals() else 0,
    "notebook_generated_smoke_config_candidate_count": int(len(notebook_generated_smoke_config_df)) if "notebook_generated_smoke_config_df" in globals() else 0,
    "command_preview": " ".join(shlex.quote(part) for part in campaign_smoke_command) if campaign_smoke_command else None,
    "can_build_preview": can_build_native_campaign_smoke_preview,
    "can_execute": can_execute_native_campaign_smoke,
    "status": "native_campaign_smoke_preview_only",
    "executed": False,
    "returncode": None,
}

if RUN_NATIVE_CAMPAIGN_SMOKE:
    if not native_campaign_cli_available:
        native_campaign_smoke_result["status"] = "native_campaign_smoke_blocked_cli_unavailable"
    elif preview_campaign_smoke_config_path is None:
        native_campaign_smoke_result["status"] = "native_campaign_smoke_blocked_missing_preview_config"
    elif campaign_smoke_command is None:
        native_campaign_smoke_result["status"] = "native_campaign_smoke_blocked_command_not_constructed"
    elif not ALLOW_NATIVE_CAMPAIGN_SMOKE_EXECUTION:
        # Preview profile success path: execution is intentionally disabled, but
        # the reviewable native command is available.
        native_campaign_smoke_result["status"] = "native_campaign_smoke_preview_ready"
    elif campaign_smoke_config_status in {"provisional_campaign_smoke_config_validation_failed", "notebook12_generated_smoke_config_validation_failed"}:
        native_campaign_smoke_result["status"] = "native_campaign_smoke_blocked_provisional_config_validation_failed"
    elif executable_campaign_smoke_config_path is None:
        native_campaign_smoke_result["status"] = "native_campaign_smoke_blocked_missing_executable_config"
    elif not NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY and not ALLOW_NON_DRY_RUN_NATIVE_CAMPAIGN_SMOKE_EXECUTION:
        native_campaign_smoke_result["status"] = "native_campaign_smoke_blocked_non_dry_run_execution_not_allowed"
    elif NATIVE_CAMPAIGN_SMOKE_DRY_RUN_ONLY and REQUIRE_DRY_RUN_ARGUMENT_FOR_NATIVE_CAMPAIGN_SMOKE and not selected_dry_run_option:
        native_campaign_smoke_result["status"] = "native_campaign_smoke_blocked_dry_run_unavailable"
    else:
        try:
            result = subprocess.run(
                campaign_smoke_command,
                cwd=str(STRATLAKE_ROOT),
                capture_output=True,
                text=True,
                timeout=180,
                check=False,
            )
            native_campaign_smoke_result.update({
                "executed": True,
                "returncode": result.returncode,
                "stdout_head": result.stdout[:2000],
                "stderr_head": result.stderr[:2000],
                "status": "native_campaign_smoke_executed_returncode_zero" if result.returncode == 0 else "native_campaign_smoke_executed_returncode_nonzero",
            })
        except Exception as exc:
            native_campaign_smoke_result.update({
                "executed": False,
                "status": "native_campaign_smoke_execution_error",
                "error": repr(exc),
            })

print("Native campaign config candidates")
show_df(native_campaign_template_df, n=20)
print("\nNative campaign smoke result")
print_json(native_campaign_smoke_result)


## 9. Optional StratLake archive restore

Restore is preview-only by default.

Use archive restore only when `RUN_STRATLAKE_ARCHIVE_RESTORE = True` and the archive target has been manually reviewed.


In [ ]:
STRATLAKE_RESTORE_ARCHIVE_ID = "REPLACE_WITH_CAMPAIGN_OR_NOTEBOOK11_ARCHIVE_ID"
STRATLAKE_RESTORE_DRIVE_ROOT = STRATLAKE_SESSION_ARCHIVE_ROOT.as_posix()

restore_command = [
    "stratlake-session-archive-restore-bootstrap",
    "--root", str(STRATLAKE_ROOT),
    "--archive-id", STRATLAKE_RESTORE_ARCHIVE_ID,
    "--drive-root", STRATLAKE_RESTORE_DRIVE_ROOT,
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

restore_result = {
    "command_preview": " ".join(shlex.quote(part) for part in restore_command),
    "run_requested": RUN_STRATLAKE_ARCHIVE_RESTORE,
    "status": "restore_preview_only",
}

if RUN_STRATLAKE_ARCHIVE_RESTORE:
    if command_available(restore_command[0]):
        try:
            result = subprocess.run(restore_command, capture_output=True, text=True, timeout=300, check=False)
            restore_result.update({
                "returncode": result.returncode,
                "stdout_head": result.stdout[:2000],
                "stderr_head": result.stderr[:2000],
                "status": "stratlake_archive_restore_completed" if result.returncode == 0 else "stratlake_archive_restore_returned_nonzero",
            })
        except Exception as exc:
            restore_result.update({"status": "stratlake_archive_restore_error", "error": repr(exc)})
    else:
        restore_result.update({"status": "stratlake_archive_restore_cli_unavailable"})

print_json(restore_result)


## 10. Notebook 11 context discovery

Notebook 11 context is optional. If enabled, this cell looks for prior review outputs and records whether expanded-run context was loaded.

Notebook 11's four manual-review candidates are **not** converted into campaign rows unless a clear campaign/run registry maps them.


In [ ]:
NOTEBOOK11_EXPECTED_FILES = [
    "summary.json",
    "final_handoff.json",
    "promotion_evidence_review.csv",
    "promotion_evidence_review.json",
    "expanded_execution_results.csv",
    "expanded_metric_evidence.csv",
    "caveat_register.csv",
]


def read_json_file(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def read_table_file(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".json", ".jsonl"}:
        try:
            return pd.read_json(path)
        except ValueError:
            data = read_json_file(path)
            if isinstance(data, list):
                return pd.DataFrame(data)
            if isinstance(data, dict):
                rows = data.get("rows") or data.get("records") or data.get("data")
                if isinstance(rows, list):
                    return pd.DataFrame(rows)
                return pd.DataFrame([data])
    return pd.DataFrame()


def load_expected_context(root: Path, filenames: list[str]) -> dict[str, Any]:
    loaded: dict[str, Any] = {}
    for filename in filenames:
        path = root / filename
        if not path.exists():
            continue
        try:
            if path.suffix.lower() == ".csv":
                loaded[filename] = pd.read_csv(path)
            else:
                loaded[filename] = read_json_file(path)
        except Exception as exc:
            loaded[filename] = {"load_error": repr(exc), "path": str(path)}
    return loaded


notebook11_context = {}
notebook11_context_status = "notebook11_context_skipped"

if DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT:
    notebook11_context = load_expected_context(NOTEBOOK11_REVIEW_DIR, NOTEBOOK11_EXPECTED_FILES)
    if notebook11_context:
        notebook11_context_status = "notebook11_context_loaded"
        if "expanded_execution_results.csv" in notebook11_context or "expanded_metric_evidence.csv" in notebook11_context:
            notebook11_context_status = "notebook11_expanded_run_context_loaded"
    else:
        notebook11_context_status = "notebook11_context_missing"

notebook11_context_summary = {
    "discover_requested": DISCOVER_NOTEBOOK11_EXPANDED_CONTEXT,
    "notebook11_review_dir": str(NOTEBOOK11_REVIEW_DIR),
    "loaded_files": list(notebook11_context.keys()),
    "status": notebook11_context_status,
}
print_json(notebook11_context_summary)



## 11. Campaign artifact discovery

Campaign artifact discovery is off by default and strict by design.

This cell searches a bounded set of likely restored artifact roots for known campaign artifact names. It also builds a candidate campaign-context inventory grouped by parent directory, inferred campaign id, and inferred run id. This helps the reviewer decide whether restored artifacts are coherent enough to load.

Strict campaign/run ID filtering is preferred. If no filters are provided, the notebook records caveats and avoids treating discovered artifacts as promotion-grade evidence. Notebook-generated review files under the Notebook 12 review directory are surfaced separately as `notebook12_review_artifact` and do **not** count as native campaign context.


In [ ]:

import re

KNOWN_CAMPAIGN_ARTIFACT_NAMES = [
    "campaign_manifest.json",
    "campaign_manifest.csv",
    "campaign_run_registry.json",
    "campaign_run_registry.csv",
    "campaign_summary.json",
    "campaign_summary.csv",
    "campaign_decision_log.json",
    "campaign_decision_log.csv",
    "campaign_artifact_inventory.json",
    "campaign_artifact_inventory.csv",
    "campaign_evidence_review.json",
    "campaign_evidence_review.csv",
    "campaign_report.md",
    "research_campaign_summary.json",
    "research_campaign_registry.csv",
]

# Native campaign context should be established by platform/campaign markers,
# not by this notebook's own generated review files.
NATIVE_CAMPAIGN_MARKER_NAMES = {
    "campaign_manifest.json",
    "campaign_manifest.csv",
    "campaign_run_registry.json",
    "campaign_run_registry.csv",
    "campaign_summary.json",
    "campaign_summary.csv",
    "research_campaign_summary.json",
    "research_campaign_registry.csv",
}

NOTEBOOK12_GENERATED_REVIEW_NAMES = {
    "campaign_artifact_inventory.csv",
    "campaign_evidence_review.csv",
    "summary.json",
    "final_handoff.json",
    "caveat_register.csv",
    "candidate_campaign_context_inventory.csv",
    "native_campaign_smoke_result.json",
}

CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS = [
    CAMPAIGN_ARTIFACT_ROOT,
    STRATLAKE_ROOT / "campaigns",
    STRATLAKE_ROOT / "runs",
    STRATLAKE_ROOT / "reports",
]
if CAMPAIGN_DISCOVERY_INCLUDE_NOTEBOOK11_REVIEW_DIR:
    CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS.append(NOTEBOOK11_REVIEW_DIR)
if CAMPAIGN_DISCOVERY_INCLUDE_NOTEBOOK12_REVIEW_DIR:
    CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS.append(NOTEBOOK12_REVIEW_DIR)

# Keep discovery roots unique while preserving order.
_seen_discovery_roots = set()
CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS = [
    root for root in CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS
    if not (root.as_posix() in _seen_discovery_roots or _seen_discovery_roots.add(root.as_posix()))
]

CAMPAIGN_ID_PATTERN = re.compile(r"(?:campaign|research_campaign)[_-][A-Za-z0-9._-]+")
RUN_ID_PATTERN = re.compile(r"(?:run|strategy_run)[_-][A-Za-z0-9._-]+")


def infer_identifier_from_path(path: Path, pattern: re.Pattern[str]) -> str | None:
    text = path.as_posix()
    matches = pattern.findall(text)
    if not matches:
        return None
    # Prefer the most specific-looking match, usually the longest path token.
    return sorted(matches, key=len, reverse=True)[0].rstrip("./_")


def path_is_relative_to(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except Exception:
        return False


def path_matches_strict_filters(path: Path, campaign_id: str | None, run_id: str | None) -> bool:
    text = str(path)
    if campaign_id and campaign_id not in text:
        return False
    if run_id and run_id not in text:
        return False
    return True


def within_max_depth(path: Path, root: Path, max_depth: int) -> bool:
    try:
        relative_parts = path.relative_to(root).parts
    except ValueError:
        return True
    return len(relative_parts) <= max_depth


def classify_artifact_origin(path: Path) -> str:
    """Classify discovered files so notebook-generated review artifacts cannot
    masquerade as native campaign context.
    """
    if path_is_relative_to(path, NOTEBOOK12_REVIEW_DIR):
        return "notebook12_review_artifact"
    if path_is_relative_to(path, NOTEBOOK11_REVIEW_DIR):
        return "notebook11_review_artifact"
    return "native_campaign_artifact"


def artifact_has_native_identifier(path: Path) -> bool:
    """Lightweight content check for inventory/review files that may only be
    native campaign context when they carry campaign_id or run_id values.
    """
    try:
        suffix = path.suffix.lower()
        if suffix == ".json":
            data = read_json_file(path)
            if isinstance(data, dict):
                keys = {str(k) for k in data.keys()}
                if keys.intersection({"campaign_id", "research_campaign_id", "run_id", "runs", "registry", "scenarios"}):
                    return True
            if isinstance(data, list) and data and isinstance(data[0], dict):
                keys = {str(k) for k in data[0].keys()}
                if keys.intersection({"campaign_id", "research_campaign_id", "run_id"}):
                    return True
        elif suffix == ".csv":
            preview = pd.read_csv(path, nrows=25)
            id_cols = [c for c in preview.columns if str(c).lower() in {"campaign_id", "research_campaign_id", "run_id"}]
            for col in id_cols:
                values = preview[col].dropna().astype(str).str.strip()
                if not values.empty and values.ne("").any():
                    return True
    except Exception:
        return False
    return False


def is_native_campaign_marker(path: Path, artifact_name: str, artifact_origin: str) -> bool:
    if artifact_origin != "native_campaign_artifact":
        return False
    if artifact_name in NATIVE_CAMPAIGN_MARKER_NAMES:
        return True
    if artifact_name in {"campaign_artifact_inventory.json", "campaign_artifact_inventory.csv", "campaign_evidence_review.json", "campaign_evidence_review.csv"}:
        return artifact_has_native_identifier(path)
    return False


def discover_campaign_artifacts(roots: list[Path]) -> pd.DataFrame:
    rows = []
    seen_paths: set[str] = set()
    for root in roots:
        if len(rows) >= CAMPAIGN_DISCOVERY_MAX_FILES:
            break
        if not root.exists():
            continue
        for name in KNOWN_CAMPAIGN_ARTIFACT_NAMES:
            if len(rows) >= CAMPAIGN_DISCOVERY_MAX_FILES:
                break
            for path in root.rglob(name):
                if len(rows) >= CAMPAIGN_DISCOVERY_MAX_FILES:
                    break
                if not within_max_depth(path, root, CAMPAIGN_DISCOVERY_MAX_DEPTH):
                    continue
                path_key = path.resolve().as_posix()
                if path_key in seen_paths:
                    continue
                strict_match = path_matches_strict_filters(path, CAMPAIGN_ID_FILTER, RUN_ID_FILTER)
                if RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY and (CAMPAIGN_ID_FILTER or RUN_ID_FILTER) and not strict_match:
                    continue
                seen_paths.add(path_key)
                stat = path.stat()
                artifact_origin = classify_artifact_origin(path)
                native_marker = is_native_campaign_marker(path, name, artifact_origin)
                inferred_campaign_id = infer_identifier_from_path(path, CAMPAIGN_ID_PATTERN)
                inferred_run_id = infer_identifier_from_path(path, RUN_ID_PATTERN)
                rows.append({
                    "artifact_name": name,
                    "path": str(path),
                    "discovery_root": str(root),
                    "parent_dir": str(path.parent),
                    "suffix": path.suffix.lower(),
                    "size_bytes": stat.st_size,
                    "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
                    "strict_match": strict_match,
                    "artifact_origin": artifact_origin,
                    "native_campaign_marker": native_marker,
                    "notebook_generated_review_artifact": artifact_origin in {"notebook12_review_artifact", "notebook11_review_artifact"},
                    "inferred_campaign_id": inferred_campaign_id,
                    "inferred_run_id": inferred_run_id,
                })
    columns = [
        "artifact_name", "path", "discovery_root", "parent_dir", "suffix", "size_bytes", "modified_utc",
        "strict_match", "artifact_origin", "native_campaign_marker", "notebook_generated_review_artifact",
        "inferred_campaign_id", "inferred_run_id",
    ]
    return pd.DataFrame(rows, columns=columns)


def build_campaign_context_candidates(inventory_df: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "candidate_context_key",
        "parent_dir",
        "inferred_campaign_id",
        "inferred_run_id",
        "artifact_count",
        "artifact_names",
        "native_campaign_marker_count",
        "metrics_like_artifacts",
        "manifest_like_artifacts",
        "governance_like_artifacts",
        "latest_modified_utc",
        "strict_match_count",
        "coherence_status",
    ]
    if inventory_df.empty:
        return pd.DataFrame(columns=columns)

    # Only native campaign artifacts can create candidate campaign context.
    native_df = inventory_df.loc[inventory_df.get("artifact_origin", pd.Series(dtype=str)).eq("native_campaign_artifact")].copy()
    if native_df.empty:
        return pd.DataFrame(columns=columns)

    rows = []
    group_cols = ["parent_dir", "inferred_campaign_id", "inferred_run_id"]
    for group_key, group in native_df.groupby(group_cols, dropna=False):
        parent_dir, inferred_campaign_id, inferred_run_id = group_key
        artifact_names = sorted(group["artifact_name"].astype(str).unique().tolist())
        artifact_text = " ".join(artifact_names).lower()
        marker_count = int(group.get("native_campaign_marker", pd.Series(dtype=bool)).fillna(False).sum())
        metrics_like = any(token in artifact_text for token in ["summary", "registry", "evidence_review"])
        manifest_like = any(token in artifact_text for token in ["manifest", "inventory"])
        governance_like = any(token in artifact_text for token in ["decision", "report"])
        strict_match_count = int(group["strict_match"].fillna(False).sum())
        if marker_count == 0:
            coherence_status = "candidate_context_missing_native_campaign_marker"
        elif RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY and (CAMPAIGN_ID_FILTER or RUN_ID_FILTER) and strict_match_count == 0:
            coherence_status = "filtered_context_not_strict_match"
        elif not inferred_campaign_id and not CAMPAIGN_ID_FILTER:
            coherence_status = "candidate_context_missing_campaign_id"
        elif len(artifact_names) < 2:
            coherence_status = "candidate_context_sparse"
        else:
            coherence_status = "candidate_context_reviewable"
        rows.append({
            "candidate_context_key": f"{parent_dir}|{inferred_campaign_id or ''}|{inferred_run_id or ''}",
            "parent_dir": parent_dir,
            "inferred_campaign_id": inferred_campaign_id,
            "inferred_run_id": inferred_run_id,
            "artifact_count": int(len(group)),
            "artifact_names": ", ".join(artifact_names),
            "native_campaign_marker_count": marker_count,
            "metrics_like_artifacts": metrics_like,
            "manifest_like_artifacts": manifest_like,
            "governance_like_artifacts": governance_like,
            "latest_modified_utc": str(group["modified_utc"].max()),
            "strict_match_count": strict_match_count,
            "coherence_status": coherence_status,
        })
    return pd.DataFrame(rows, columns=columns).sort_values(
        ["coherence_status", "native_campaign_marker_count", "artifact_count", "latest_modified_utc"],
        ascending=[True, False, False, False],
    )


if RUN_CAMPAIGN_ARTIFACT_DISCOVERY or DISCOVER_CAMPAIGN_CONTEXT:
    campaign_artifact_inventory_all_df = discover_campaign_artifacts(CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS)
    native_campaign_artifact_inventory_df = campaign_artifact_inventory_all_df.loc[
        campaign_artifact_inventory_all_df.get("artifact_origin", pd.Series(dtype=str)).eq("native_campaign_artifact")
    ].copy() if not campaign_artifact_inventory_all_df.empty else campaign_artifact_inventory_all_df.copy()
    notebook12_review_artifact_inventory_df = campaign_artifact_inventory_all_df.loc[
        campaign_artifact_inventory_all_df.get("artifact_origin", pd.Series(dtype=str)).eq("notebook12_review_artifact")
    ].copy() if not campaign_artifact_inventory_all_df.empty else campaign_artifact_inventory_all_df.copy()
    # Downstream loading and evidence construction should only consume native campaign artifacts.
    campaign_artifact_inventory_df = native_campaign_artifact_inventory_df.copy()
    candidate_campaign_context_df = build_campaign_context_candidates(campaign_artifact_inventory_all_df)
    campaign_artifact_discovery_status = (
        "campaign_artifacts_discovered" if len(candidate_campaign_context_df) else "campaign_context_missing"
    )
else:
    campaign_artifact_inventory_all_df = pd.DataFrame(columns=[
        "artifact_name", "path", "discovery_root", "parent_dir", "suffix", "size_bytes", "modified_utc",
        "strict_match", "artifact_origin", "native_campaign_marker", "notebook_generated_review_artifact",
        "inferred_campaign_id", "inferred_run_id",
    ])
    native_campaign_artifact_inventory_df = campaign_artifact_inventory_all_df.copy()
    notebook12_review_artifact_inventory_df = campaign_artifact_inventory_all_df.copy()
    campaign_artifact_inventory_df = native_campaign_artifact_inventory_df.copy()
    candidate_campaign_context_df = build_campaign_context_candidates(campaign_artifact_inventory_all_df)
    campaign_artifact_discovery_status = "campaign_artifact_discovery_skipped_source_safe_preview"

campaign_artifact_discovery_summary = {
    "discover_requested": RUN_CAMPAIGN_ARTIFACT_DISCOVERY or DISCOVER_CAMPAIGN_CONTEXT,
    "artifact_roots": [root.as_posix() for root in CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS],
    "existing_artifact_roots": [root.as_posix() for root in CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS if root.exists()],
    "total_discovered_artifact_rows": int(len(campaign_artifact_inventory_all_df)),
    "native_campaign_artifact_rows": int(len(campaign_artifact_inventory_df)),
    "notebook12_review_artifact_rows": int(len(notebook12_review_artifact_inventory_df)),
    "artifact_rows": int(len(campaign_artifact_inventory_df)),
    "candidate_context_rows": int(len(candidate_campaign_context_df)),
    "candidate_context_reviewable_rows": int((candidate_campaign_context_df.get("coherence_status", pd.Series(dtype=str)) == "candidate_context_reviewable").sum()) if not candidate_campaign_context_df.empty else 0,
    "native_campaign_marker_rows": int(campaign_artifact_inventory_df.get("native_campaign_marker", pd.Series(dtype=bool)).fillna(False).sum()) if not campaign_artifact_inventory_df.empty else 0,
    "status": campaign_artifact_discovery_status,
    "strict_discovery": RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY,
    "campaign_id_filter": CAMPAIGN_ID_FILTER,
    "run_id_filter": RUN_ID_FILTER,
    "max_files": CAMPAIGN_DISCOVERY_MAX_FILES,
}
print_json(campaign_artifact_discovery_summary)
show_df(candidate_campaign_context_df)
show_df(campaign_artifact_inventory_df)
if not notebook12_review_artifact_inventory_df.empty:
    print("Notebook-generated review artifacts were discovered but excluded from native campaign context:")
    show_df(notebook12_review_artifact_inventory_df)


## 12. Load campaign artifacts

This cell defensively loads discovered campaign artifacts into a dictionary keyed by artifact name.

Missing artifacts are not fatal. Mixed or stale artifacts should be reviewed before evidence classification.


In [ ]:
def load_campaign_artifacts(inventory_df: pd.DataFrame) -> dict[str, list[Any]]:
    loaded: dict[str, list[Any]] = {}
    for _, row in inventory_df.iterrows():
        path = Path(row["path"])
        name = row["artifact_name"]
        loaded.setdefault(name, [])
        try:
            if path.suffix.lower() == ".csv":
                loaded[name].append({"path": str(path), "data": pd.read_csv(path)})
            elif path.suffix.lower() == ".json":
                loaded[name].append({"path": str(path), "data": read_json_file(path)})
            elif path.suffix.lower() == ".md":
                loaded[name].append({"path": str(path), "data": path.read_text(encoding="utf-8", errors="replace")})
            else:
                loaded[name].append({"path": str(path), "load_error": f"Unsupported suffix: {path.suffix}"})
        except Exception as exc:
            loaded[name].append({"path": str(path), "load_error": repr(exc)})
    return loaded


campaign_artifacts = load_campaign_artifacts(campaign_artifact_inventory_df)
campaign_loaded_summary = {
    "loaded_artifact_groups": list(campaign_artifacts.keys()),
    "loaded_group_count": len(campaign_artifacts),
    "loaded_item_count": sum(len(items) for items in campaign_artifacts.values()),
}
print_json(campaign_loaded_summary)


## 13. Campaign evidence dataframe schema

Minimum campaign evidence columns:

```text
campaign_id
scenario_id
strategy_id
run_id
window_name
run_status
metrics_loaded
split_metrics_loaded
promotion_gates_loaded
manifest_loaded
governance_loaded
artifact_inventory_loaded
retry_status
reuse_status
evidence_status
human_review_status
blocking_reason
caveat_count
```

If no campaign artifacts exist, this notebook creates an empty dataframe with the correct schema and records a caveat.


In [ ]:
CAMPAIGN_EVIDENCE_COLUMNS = [
    "campaign_id",
    "scenario_id",
    "strategy_id",
    "run_id",
    "window_name",
    "run_status",
    "metrics_loaded",
    "split_metrics_loaded",
    "promotion_gates_loaded",
    "manifest_loaded",
    "governance_loaded",
    "artifact_inventory_loaded",
    "retry_status",
    "reuse_status",
    "evidence_status",
    "human_review_status",
    "blocking_reason",
    "caveat_count",
]


def records_from_loaded_artifact_items(items: list[Any]) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for item in items:
        data = item.get("data") if isinstance(item, dict) else item
        if isinstance(data, pd.DataFrame):
            records.extend(data.to_dict(orient="records"))
        elif isinstance(data, list):
            records.extend([row for row in data if isinstance(row, dict)])
        elif isinstance(data, dict):
            for key in ["rows", "records", "data", "runs", "scenarios", "registry"]:
                value = data.get(key)
                if isinstance(value, list):
                    records.extend([row for row in value if isinstance(row, dict)])
                    break
            else:
                records.append(data)
    return records


def coalesce(row: dict[str, Any], *names: str, default: Any = None) -> Any:
    for name in names:
        value = row.get(name)
        if value not in [None, ""]:
            return value
    return default


def bool_loaded_from_names(row: dict[str, Any], *names: str) -> bool:
    for name in names:
        value = row.get(name)
        if isinstance(value, bool):
            if value:
                return True
        elif value not in [None, "", 0, "0", False]:
            return True
    return False


def normalize_campaign_record(row: dict[str, Any], source_name: str) -> dict[str, Any]:
    return {
        "campaign_id": coalesce(row, "campaign_id", "campaign", "research_campaign_id", default=None),
        "scenario_id": coalesce(row, "scenario_id", "scenario", "config_id", "case_id", default=None),
        "strategy_id": coalesce(row, "strategy_id", "strategy", "strategy_name", default=None),
        "run_id": coalesce(row, "run_id", "execution_id", "strategy_run_id", default=None),
        "window_name": coalesce(row, "window_name", "window", "split_name", "period", default=None),
        "run_status": coalesce(row, "run_status", "status", "execution_status", default="unknown"),
        "metrics_loaded": bool_loaded_from_names(row, "metrics_loaded", "metrics_path", "metrics_file", "metrics"),
        "split_metrics_loaded": bool_loaded_from_names(row, "split_metrics_loaded", "split_metrics_path", "split_metrics_file", "split_metrics"),
        "promotion_gates_loaded": bool_loaded_from_names(row, "promotion_gates_loaded", "promotion_gates_path", "promotion_gates_file", "promotion_gates"),
        "manifest_loaded": bool_loaded_from_names(row, "manifest_loaded", "manifest_path", "manifest_file") or source_name in {"campaign_manifest.json", "campaign_manifest.csv"},
        "governance_loaded": bool_loaded_from_names(row, "governance_loaded", "governance_report_path", "governance_report_file"),
        "artifact_inventory_loaded": bool_loaded_from_names(row, "artifact_inventory_loaded", "artifact_inventory_path", "artifact_inventory_file") or source_name in {"campaign_artifact_inventory.json", "campaign_artifact_inventory.csv"},
        "retry_status": coalesce(row, "retry_status", "retry", "checkpoint_retry_status", default="unknown"),
        "reuse_status": coalesce(row, "reuse_status", "reuse", "reuse_policy_status", default="unknown"),
        "evidence_status": coalesce(row, "evidence_status", default="unclassified"),
        "human_review_status": coalesce(row, "human_review_status", default="not_reviewed"),
        "blocking_reason": coalesce(row, "blocking_reason", "blocker", default=None),
        "caveat_count": int(coalesce(row, "caveat_count", default=0) or 0),
    }


candidate_records: list[dict[str, Any]] = []

for source_name in [
    "campaign_evidence_review.csv",
    "campaign_evidence_review.json",
    "campaign_run_registry.csv",
    "campaign_run_registry.json",
    "research_campaign_registry.csv",
    "campaign_manifest.csv",
    "campaign_manifest.json",
]:
    for record in records_from_loaded_artifact_items(campaign_artifacts.get(source_name, [])):
        candidate_records.append(normalize_campaign_record(record, source_name))

campaign_evidence_df = pd.DataFrame(candidate_records, columns=CAMPAIGN_EVIDENCE_COLUMNS)

if campaign_evidence_df.empty:
    campaign_evidence_df = pd.DataFrame(columns=CAMPAIGN_EVIDENCE_COLUMNS)

print(f"Campaign evidence rows: {len(campaign_evidence_df)}")
show_df(campaign_evidence_df)


## 14. Evidence sufficiency classification

Classification is conservative and review-oriented.

Allowed review statuses include:

- `complete_platform_evidence_available`
- `needs_split_metrics`
- `needs_promotion_gates`
- `needs_manifest`
- `needs_retry`
- `needs_more_evidence`
- `blocked_failed_run`
- `blocked_missing_artifacts`
- `ready_for_human_watchlist_review`
- `not_promotion_grade`

Forbidden as claims:

- `approved`
- `promoted`
- `alpha_confirmed`
- `production_ready`
- `promotion_grade`


In [ ]:
FAILED_STATUSES = {"failed", "error", "errored", "crashed", "timeout", "cancelled", "canceled"}
SUCCESS_STATUSES = {"success", "completed", "complete", "passed", "ok"}


def classify_campaign_evidence(row: pd.Series) -> pd.Series:
    run_status = str(row.get("run_status", "")).strip().lower()
    metrics_loaded = bool(row.get("metrics_loaded"))
    split_metrics_loaded = bool(row.get("split_metrics_loaded"))
    promotion_gates_loaded = bool(row.get("promotion_gates_loaded"))
    manifest_loaded = bool(row.get("manifest_loaded"))
    governance_loaded = bool(row.get("governance_loaded"))
    caveat_count = int(row.get("caveat_count") or 0)

    evidence_status = "needs_more_evidence"
    human_review_status = "not_ready_for_human_watchlist_review"
    blocking_reason = row.get("blocking_reason")

    if run_status in FAILED_STATUSES:
        evidence_status = "blocked_failed_run"
        blocking_reason = blocking_reason or "run_status_failed_or_error"
    elif not metrics_loaded and not manifest_loaded:
        evidence_status = "blocked_missing_artifacts"
        blocking_reason = blocking_reason or "no_metrics_or_manifest_loaded"
    elif metrics_loaded and not split_metrics_loaded:
        evidence_status = "needs_split_metrics"
        blocking_reason = blocking_reason or "split_metrics_missing"
    elif metrics_loaded and split_metrics_loaded and not promotion_gates_loaded:
        evidence_status = "needs_promotion_gates"
        blocking_reason = blocking_reason or "promotion_gates_missing"
    elif not manifest_loaded:
        evidence_status = "needs_manifest"
        blocking_reason = blocking_reason or "manifest_missing"
    elif metrics_loaded and split_metrics_loaded and promotion_gates_loaded and manifest_loaded:
        if governance_loaded or not RUN_CAMPAIGN_GOVERNANCE_REVIEW:
            evidence_status = "ready_for_human_watchlist_review"
            human_review_status = "ready_for_human_watchlist_review" if caveat_count == 0 else "human_review_with_caveats"
            blocking_reason = blocking_reason or None
        else:
            evidence_status = "needs_more_evidence"
            blocking_reason = blocking_reason or "governance_report_missing"

    row["evidence_status"] = evidence_status
    row["human_review_status"] = human_review_status
    row["blocking_reason"] = blocking_reason
    return row


if not campaign_evidence_df.empty:
    campaign_evidence_df = campaign_evidence_df.apply(classify_campaign_evidence, axis=1)

evidence_status_counts = (
    campaign_evidence_df["evidence_status"].value_counts(dropna=False).to_dict()
    if "evidence_status" in campaign_evidence_df.columns
    else {}
)

print_json({"evidence_status_counts": evidence_status_counts})
show_df(campaign_evidence_df)


## 15. Governance, evidence, campaign execution, and report CLI surfaces

Capture root and subcommand help for native StratLake campaign-oriented CLIs.  The notebook treats CLI help as the source of truth for command shape, argument names, and whether a runtime command can be previewed safely.  Missing or unclear surfaces become caveats, not notebook failures.


In [ ]:
REPORT_AND_GOVERNANCE_COMMANDS = [
    "stratlake-run-research-campaign",
    "stratlake-build-campaign-report",
    "stratlake-build-evidence-review",
    "stratlake-run-promotion-governance-report",
]

CAMPAIGN_COMMAND_HELP_CANDIDATE_SUBCOMMANDS = {
    "stratlake-run-research-campaign": ["run", "build", "execute", "inspect", "validate"],
    "stratlake-build-campaign-report": ["build", "report", "summarize", "inspect", "validate"],
    "stratlake-build-evidence-review": ["build", "review", "report", "summarize", "inspect", "validate"],
    "stratlake-run-promotion-governance-report": ["run", "build", "report", "summarize", "inspect", "validate"],
}

ARGUMENT_ALIAS_GROUPS = {
    "root": ["--root", "--workspace-root", "--project-root", "--stratlake-root"],
    "campaign_id": ["--campaign-id", "--campaign_id", "--campaign", "--campaign-name"],
    "campaign_config": ["--campaign-config", "--campaign_config", "--config", "--config-path", "--campaign-plan", "--plan"],
    "campaign_root": ["--campaign-root", "--campaign_root", "--campaign-dir", "--campaign-path"],
    "artifact_root": ["--artifact-root", "--artifact_root", "--artifacts-root", "--artifacts_root", "--artifacts-dir", "--artifacts", "--artifact-dir"],
    "input": ["--input", "--input-path", "--manifest", "--campaign-manifest", "--registry", "--run-registry"],
    "output": ["--output", "--output-path", "--output-dir", "--review-dir", "--report-dir", "--out"],
    "format": ["--format", "--output-format"],
}

ARGUMENT_VALUE_BY_GROUP = {
    "root": str(STRATLAKE_ROOT),
    "campaign_id": "REPLACE_WITH_CAMPAIGN_ID",
    "campaign_config": "REPLACE_WITH_CAMPAIGN_CONFIG.yml",
    "campaign_root": str(CAMPAIGN_ARTIFACT_ROOT),
    "artifact_root": str(CAMPAIGN_ARTIFACT_ROOT),
    "input": "REPLACE_WITH_CAMPAIGN_MANIFEST_OR_REGISTRY_PATH",
    "output": str(NOTEBOOK12_REVIEW_DIR),
    "format": "json",
}


def command_help_text(command_parts: list[str], timeout_seconds: int = 20, max_chars: int = 20000) -> dict[str, Any]:
    """Capture CLI help for a root command or a subcommand without failing the notebook."""
    root_command = command_parts[0]
    if not command_available(root_command):
        return {
            "command": " ".join(command_parts),
            "root_command": root_command,
            "available": False,
            "help_checked": False,
            "returncode": None,
            "stdout": "",
            "stderr": "",
            "status": "cli_unavailable",
        }
    try:
        result = subprocess.run(
            [*command_parts, "--help"],
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            check=False,
        )
        stdout = result.stdout or ""
        stderr = result.stderr or ""
        return {
            "command": " ".join(command_parts),
            "root_command": root_command,
            "available": True,
            "help_checked": True,
            "returncode": result.returncode,
            "stdout": stdout[:max_chars],
            "stderr": stderr[:max_chars],
            "status": "cli_help_loaded" if result.returncode == 0 else "cli_help_returned_nonzero",
        }
    except Exception as exc:
        return {
            "command": " ".join(command_parts),
            "root_command": root_command,
            "available": True,
            "help_checked": False,
            "returncode": None,
            "stdout": "",
            "stderr": "",
            "error": repr(exc),
            "status": "cli_help_error",
        }


def help_combined_text(help_record: dict[str, Any] | pd.Series) -> str:
    return f"{help_record.get('stdout', '')}\n{help_record.get('stderr', '')}"


def help_combined_text_lower(help_record: dict[str, Any] | pd.Series) -> str:
    return help_combined_text(help_record).lower()


def help_advertises_any_term(help_record: dict[str, Any] | pd.Series, terms: list[str]) -> bool:
    text = help_combined_text_lower(help_record)
    return any(term.lower() in text for term in terms)


def parse_advertised_subcommands(help_record: dict[str, Any] | pd.Series) -> list[str]:
    """Best-effort argparse subcommand parser for help fragments like {build,review}."""
    import re

    text = help_combined_text_lower(help_record)
    discovered: set[str] = set()
    for match in re.findall(r"\{([^{}]+)\}", text):
        for part in match.split(","):
            candidate = part.strip().strip("'").strip('"')
            if candidate and all(ch.isalnum() or ch in "_-" for ch in candidate):
                discovered.add(candidate)
    return sorted(discovered)


def parse_advertised_options(help_record: dict[str, Any] | pd.Series) -> list[str]:
    """Extract long-option names from captured CLI help."""
    import re

    text = help_combined_text(help_record)
    options = sorted(set(re.findall(r"(?<![\w-])--[A-Za-z][A-Za-z0-9_-]*", text)))
    return options


def argument_groups_from_options(options: list[str]) -> dict[str, list[str]]:
    option_set = set(options)
    detected: dict[str, list[str]] = {}
    for group, aliases in ARGUMENT_ALIAS_GROUPS.items():
        matched = [alias for alias in aliases if alias in option_set]
        if matched:
            detected[group] = matched
    return detected


def record_argument_surface(help_record: dict[str, Any]) -> dict[str, Any]:
    options = parse_advertised_options(help_record)
    groups = argument_groups_from_options(options)
    help_record["advertised_options"] = ",".join(options)
    help_record["advertised_option_count"] = len(options)
    help_record["detected_argument_groups"] = ",".join(sorted(groups))
    for group in ARGUMENT_ALIAS_GROUPS:
        help_record[f"advertises_{group}_argument"] = group in groups
    # Backward-compatible columns used by earlier draft logic/audits.
    help_record["advertises_root_argument"] = "root" in groups
    help_record["advertises_campaign_id_argument"] = "campaign_id" in groups
    return help_record


root_help_records = []
subcommand_help_records = []

for command in REPORT_AND_GOVERNANCE_COMMANDS:
    root_help = command_help_text([command])
    root_help["help_scope"] = "root"
    root_help["subcommand"] = None
    advertised_subcommands = parse_advertised_subcommands(root_help)
    root_help["advertised_subcommands"] = ",".join(advertised_subcommands)
    root_help = record_argument_surface(root_help)
    root_help_records.append(root_help)

    candidate_subcommands = sorted(set(advertised_subcommands) | set(CAMPAIGN_COMMAND_HELP_CANDIDATE_SUBCOMMANDS.get(command, [])))
    if root_help.get("available"):
        for subcommand in candidate_subcommands:
            sub_help = command_help_text([command, subcommand])
            sub_help["help_scope"] = "subcommand"
            sub_help["subcommand"] = subcommand
            sub_help["advertised_subcommands"] = ""
            sub_help = record_argument_surface(sub_help)
            # Keep valid help and useful nonzero help rows that reveal arguments.
            if sub_help.get("returncode") == 0 or sub_help.get("advertised_option_count", 0) > 0:
                subcommand_help_records.append(sub_help)

report_governance_help_df = pd.DataFrame(root_help_records + subcommand_help_records)

if not report_governance_help_df.empty:
    def classify_help_argument_surface(row: pd.Series) -> str:
        if not bool(row.get("available")):
            return "cli_unavailable"
        groups = set(str(row.get("detected_argument_groups", "")).split(",")) - {""}
        if {"root", "campaign_id"}.issubset(groups):
            return "root_and_campaign_id_arguments"
        if {"campaign_root", "output"}.issubset(groups) or {"artifact_root", "output"}.issubset(groups):
            return "artifact_root_and_output_arguments"
        if {"campaign_config"}.issubset(groups) and ("root" in groups or "campaign_root" in groups):
            return "campaign_config_execution_arguments"
        if groups:
            return "partial_argument_surface_detected"
        return "installed_cli_help_does_not_advertise_known_arguments"

    report_governance_help_df["argument_surface_status"] = report_governance_help_df.apply(classify_help_argument_surface, axis=1)

help_display_columns = [
    "command",
    "help_scope",
    "subcommand",
    "available",
    "returncode",
    "advertised_option_count",
    "detected_argument_groups",
    "advertised_subcommands",
    "argument_surface_status",
]
show_df(report_governance_help_df[[c for c in help_display_columns if c in report_governance_help_df.columns]], n=50)


## 16. Optional campaign evidence/review command previews

Build native command previews only from the argument aliases actually advertised by CLI help. Runtime execution remains off by default. This section now separates three concepts that should not be conflated: detected-safe command shape, execution flag enabled, and safe-to-execute status. In preview mode, a command may have a safe detected shape while still being intentionally blocked from execution.


In [ ]:
CAMPAIGN_COMMAND_SPECS = {
    "run_research_campaign": {
        "command": "stratlake-run-research-campaign",
        "preferred_subcommands": ["run", "execute", "build"],
        "required_argument_groups_any": [
            ["campaign_config"],
            ["root", "campaign_config"],
            ["campaign_root", "campaign_config"],
        ],
        "preferred_argument_groups": ["root", "campaign_config", "campaign_id", "output"],
        "execution_enabled": NOTEBOOK12_MODE == "campaign_evidence_run",
    },
    "build_campaign_report": {
        "command": "stratlake-build-campaign-report",
        "preferred_subcommands": ["build", "report", "summarize"],
        "required_argument_groups_any": [
            ["root", "campaign_id"],
            ["campaign_root", "output"],
            ["artifact_root", "output"],
            ["input", "output"],
        ],
        "preferred_argument_groups": ["root", "campaign_id", "campaign_root", "artifact_root", "input", "output", "format"],
        "execution_enabled": RUN_STRATLAKE_CAMPAIGN_REPORT,
    },
    "build_evidence_review": {
        "command": "stratlake-build-evidence-review",
        "preferred_subcommands": ["build", "review", "report", "summarize", "validate"],
        "required_argument_groups_any": [
            ["root", "campaign_id"],
            ["campaign_root", "output"],
            ["artifact_root", "output"],
            ["input", "output"],
            ["campaign_root"],
            ["artifact_root"],
        ],
        "preferred_argument_groups": ["root", "campaign_id", "campaign_root", "artifact_root", "input", "output", "format"],
        "execution_enabled": RUN_CAMPAIGN_EVIDENCE_REVIEW,
    },
    "run_promotion_governance_report": {
        "command": "stratlake-run-promotion-governance-report",
        "preferred_subcommands": ["run", "build", "report", "summarize", "inspect", "validate"],
        "required_argument_groups_any": [
            ["root", "campaign_id"],
            ["campaign_root", "output"],
            ["artifact_root", "output"],
            ["input", "output"],
            ["campaign_root"],
            ["artifact_root"],
        ],
        "preferred_argument_groups": ["root", "campaign_id", "campaign_root", "artifact_root", "input", "output", "format"],
        "execution_enabled": RUN_CAMPAIGN_GOVERNANCE_REVIEW,
    },
}


def get_help_rows_for_command(command: str) -> pd.DataFrame:
    if "report_governance_help_df" not in globals() or report_governance_help_df.empty:
        return pd.DataFrame()
    return report_governance_help_df.loc[report_governance_help_df["root_command"] == command].copy()


def detected_groups_from_help_row(row: pd.Series) -> dict[str, list[str]]:
    options = str(row.get("advertised_options", "")).split(",") if row.get("advertised_options") is not None else []
    return argument_groups_from_options([option for option in options if option])


def selected_alias_for_group(groups: dict[str, list[str]], group: str) -> str | None:
    aliases = groups.get(group, [])
    if not aliases:
        return None
    preferred_order = ARGUMENT_ALIAS_GROUPS.get(group, [])
    for alias in preferred_order:
        if alias in aliases:
            return alias
    return aliases[0]


def row_satisfies_argument_requirements(groups: dict[str, list[str]], requirements_any: list[list[str]]) -> bool:
    if not requirements_any:
        return bool(groups)
    group_names = set(groups)
    return any(set(requirement).issubset(group_names) for requirement in requirements_any)


def build_args_from_detected_groups(groups: dict[str, list[str]], preferred_groups: list[str]) -> list[str]:
    args: list[str] = []
    for group in preferred_groups:
        alias = selected_alias_for_group(groups, group)
        if not alias:
            continue
        value = ARGUMENT_VALUE_BY_GROUP.get(group)
        if value is None:
            continue
        args.extend([alias, str(value)])
    return args


def select_command_shape(command: str, preferred_subcommands: list[str], requirements_any: list[list[str]]) -> dict[str, Any]:
    """Select the safest command shape from captured help data."""
    if not command_available(command):
        return {
            "command_shape_status": "cli_unavailable",
            "selected_subcommand": None,
            "detected_argument_groups": {},
            "reason": "Root command is not installed or not on PATH.",
        }

    rows = get_help_rows_for_command(command)
    if rows.empty:
        return {
            "command_shape_status": "direct_preview_help_not_captured",
            "selected_subcommand": None,
            "detected_argument_groups": {},
            "reason": "Help was not captured for this command; direct preview only, not safe for execution.",
        }

    root_rows = rows.loc[rows["help_scope"] == "root"].copy()
    if not root_rows.empty:
        root_row = root_rows.iloc[0]
        groups = detected_groups_from_help_row(root_row)
        if row_satisfies_argument_requirements(groups, requirements_any):
            return {
                "command_shape_status": "direct_root_args",
                "selected_subcommand": None,
                "detected_argument_groups": groups,
                "reason": "Root help advertises a minimally usable argument surface.",
            }

    sub_rows = rows.loc[rows["help_scope"] == "subcommand"].copy()
    if not sub_rows.empty:
        sub_rows["preferred_rank"] = sub_rows["subcommand"].apply(
            lambda value: preferred_subcommands.index(value) if value in preferred_subcommands else 999
        )
        candidates = []
        for _, row in sub_rows.sort_values(["preferred_rank", "subcommand"]).iterrows():
            groups = detected_groups_from_help_row(row)
            if row_satisfies_argument_requirements(groups, requirements_any):
                candidates.append((row, groups))
        if candidates:
            selected, groups = candidates[0]
            return {
                "command_shape_status": "subcommand_args",
                "selected_subcommand": selected.get("subcommand"),
                "detected_argument_groups": groups,
                "reason": "Subcommand help advertises a minimally usable argument surface.",
            }

    return {
        "command_shape_status": "argument_surface_unclear",
        "selected_subcommand": None,
        "detected_argument_groups": {},
        "reason": "Installed CLI help did not advertise a minimally usable root or subcommand argument surface.",
    }


def build_native_campaign_command_preview(spec: dict[str, Any]) -> dict[str, Any]:
    command = spec["command"]
    shape = select_command_shape(
        command,
        spec.get("preferred_subcommands", []),
        spec.get("required_argument_groups_any", []),
    )
    command_parts = [command]
    selected_subcommand = shape.get("selected_subcommand")
    if shape.get("command_shape_status") == "subcommand_args" and selected_subcommand:
        command_parts.append(selected_subcommand)

    detected_groups = shape.get("detected_argument_groups", {}) or {}
    detected_args = build_args_from_detected_groups(detected_groups, spec.get("preferred_argument_groups", []))
    command_parts.extend(detected_args)

    shape_is_safe = shape.get("command_shape_status") in {"direct_root_args", "subcommand_args"} and bool(detected_args)
    execution_enabled = bool(spec.get("execution_enabled"))
    safe_to_execute = execution_enabled and shape_is_safe

    if shape_is_safe and execution_enabled:
        execution_readiness_status = "shape_safe_execution_enabled"
    elif shape_is_safe and not execution_enabled:
        execution_readiness_status = "shape_safe_execution_disabled_preview"
    elif execution_enabled and not shape_is_safe:
        execution_readiness_status = "execution_enabled_but_shape_unclear"
    else:
        execution_readiness_status = "shape_unclear_execution_disabled"

    return {
        "command": " ".join(shlex.quote(str(part)) for part in command_parts),
        "available": command_available(command),
        "execution_enabled": execution_enabled,
        "shape_safe_for_preview": shape_is_safe,
        "safe_to_execute": safe_to_execute,
        # Backward-compatible name retained, but now means true runtime readiness, not preview shape safety.
        "safe_to_execute_from_detected_shape": safe_to_execute,
        "execution_readiness_status": execution_readiness_status,
        "detected_argument_group_names": ",".join(sorted(detected_groups)),
        "detected_argument_count": int(len(detected_args) // 2),
        **{key: value for key, value in shape.items() if key != "detected_argument_groups"},
    }


native_campaign_preview_rows = []
for surface, spec in CAMPAIGN_COMMAND_SPECS.items():
    native_campaign_preview_rows.append({
        "surface": surface,
        **build_native_campaign_command_preview(spec),
    })

native_campaign_preview_df = pd.DataFrame(native_campaign_preview_rows)
show_df(native_campaign_preview_df, n=20)

# Keep a machine-readable copy for summary and handoff inspection.
native_campaign_command_previews = native_campaign_preview_df.to_dict(orient="records")


## 17. Caveat register

The caveat register is explicit and conservative. It records missing campaign context, missing platform surfaces, source-safe previews, and non-claims.


In [ ]:
CAVEAT_COLUMNS = ["category", "severity", "message", "created_at_utc"]

caveats: list[dict[str, Any]] = []


def add_caveat(category: str, severity: str, message: str) -> None:
    caveats.append({
        "category": category,
        "severity": severity,
        "message": message,
        "created_at_utc": utc_now_iso(),
    })


if SESSION_ARCHIVE_ROOT_DISCOVERY.get("status") == "session_archive_root_default_selected":
    add_caveat(
        "session_archive_root_discovery",
        "info",
        f"No existing session archive root discovered; using default: {STRATLAKE_SESSION_ARCHIVE_ROOT.as_posix()}."
    )
else:
    add_caveat(
        "session_archive_root_discovery",
        "info",
        f"Selected discovered session archive root: {STRATLAKE_SESSION_ARCHIVE_ROOT.as_posix()}."
    )

if SOURCE_SAFE_DEFAULTS_ACTIVE:
    add_caveat("source_safe_preview", "info", "Notebook 12 is running in source-safe campaign_preview mode.")

if notebook11_context_status in {"notebook11_context_missing", "notebook11_context_skipped"}:
    add_caveat("missing_notebook11_context", "review", f"Notebook 11 context status: {notebook11_context_status}.")

if campaign_artifact_discovery_status in {
    "campaign_context_missing",
    "campaign_artifact_discovery_skipped_source_safe_preview",
}:
    add_caveat("missing_campaign_context", "review", f"Campaign artifact discovery status: {campaign_artifact_discovery_status}.")

if "notebook12_review_artifact_inventory_df" in globals() and not notebook12_review_artifact_inventory_df.empty:
    add_caveat(
        "notebook12_review_artifacts_excluded_from_campaign_context",
        "info",
        f"Excluded {len(notebook12_review_artifact_inventory_df)} Notebook 12-generated review artifact(s) from native campaign context."
    )


if "native_campaign_smoke_result" in globals() and native_campaign_smoke_result.get("config_is_notebook_generated"):
    add_caveat(
        "notebook12_generated_smoke_config_selected",
        "info",
        "Selected Notebook 12-generated smoke config; this is not being labeled as a native StratLake campaign template."
    )


if RESTORED_REVIEW_ARTIFACT_DISCOVERY_REQUESTED and REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW and not (CAMPAIGN_ID_FILTER or RUN_ID_FILTER):
    add_caveat(
        "strict_discovery_blocked",
        "blocker",
        "Restored campaign review requested, but no campaign_id or run_id filter was provided."
    )

if "candidate_campaign_context_df" in globals() and not candidate_campaign_context_df.empty:
    sparse_candidates = candidate_campaign_context_df.loc[
        candidate_campaign_context_df["coherence_status"].isin([
            "candidate_context_missing_campaign_id",
            "candidate_context_sparse",
            "candidate_context_missing_native_campaign_marker",
            "filtered_context_not_strict_match",
        ])
    ]
    for _, row in sparse_candidates.iterrows():
        add_caveat(
            "candidate_campaign_context_needs_review",
            "review",
            f"Candidate context {row.get('parent_dir')} status: {row.get('coherence_status')}."
        )

if campaign_evidence_df.empty:
    add_caveat("missing_campaign_artifact", "blocker", "No campaign evidence rows were loaded.")

if not cli_surface_df.empty:
    unavailable_commands = cli_surface_df.loc[~cli_surface_df["available"].fillna(False), "command"].tolist()
    for command in unavailable_commands:
        add_caveat("runtime_surface_unavailable", "review", f"CLI surface unavailable or not installed: {command}.")

if "native_campaign_preview_df" in globals() and not native_campaign_preview_df.empty:
    unclear_surfaces = native_campaign_preview_df.loc[
        native_campaign_preview_df["command_shape_status"].isin(["argument_surface_unclear", "cli_unavailable", "direct_preview_help_not_captured"])
    ]
    for _, row in unclear_surfaces.iterrows():
        add_caveat(
            "runtime_surface_unavailable" if row.get("command_shape_status") == "cli_unavailable" else "runtime_surface_argument_unclear",
            "review",
            f"Native campaign surface {row.get('surface')} status: {row.get('command_shape_status')} ({row.get('reason')}).",
        )

if "report_governance_help_df" in globals() and not report_governance_help_df.empty:
    partial_or_unknown = report_governance_help_df.loc[
        report_governance_help_df["argument_surface_status"].isin([
            "partial_argument_surface_detected",
            "installed_cli_help_does_not_advertise_known_arguments",
        ])
    ]
    for _, row in partial_or_unknown.iterrows():
        if row.get("available"):
            add_caveat(
                "runtime_surface_argument_partial",
                "info",
                f"CLI help for {row.get('command')} has argument surface status: {row.get('argument_surface_status')}.",
            )


if "native_campaign_smoke_result" in globals():
    smoke_status = native_campaign_smoke_result.get("status")
    if smoke_status in {
        "native_campaign_smoke_blocked_execution_not_allowed",
        "native_campaign_smoke_blocked_cli_unavailable",
        "native_campaign_smoke_blocked_missing_executable_config",
        "native_campaign_smoke_blocked_dry_run_argument_not_advertised",
        "native_campaign_smoke_blocked_command_not_constructed",
    }:
        add_caveat("native_campaign_smoke_blocked", "review", f"Native campaign smoke status: {smoke_status}.")
    elif smoke_status == "native_campaign_smoke_preview_only":
        add_caveat("native_campaign_smoke_preview", "info", "Native campaign smoke command is preview-only; execution disabled by default.")
    elif smoke_status == "native_campaign_smoke_executed_returncode_nonzero":
        add_caveat("native_campaign_smoke_execution", "blocker", "Native campaign smoke executed but returned nonzero.")
    if native_campaign_smoke_result.get("config_status") == "provisional_campaign_smoke_config_written_reference_only":
        add_caveat("reference_only_context", "info", "A provisional campaign smoke config was written for review only and was not selected for execution.")

if RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY and not (CAMPAIGN_ID_FILTER or RUN_ID_FILTER):
    add_caveat(
        "strict_discovery_blocked",
        "review",
        "Strict campaign/run artifact discovery is enabled but no campaign_id or run_id filter was provided.",
    )

# Evidence-specific caveats from loaded rows.
if not campaign_evidence_df.empty:
    for status, count in campaign_evidence_df["evidence_status"].value_counts().items():
        if status in {"needs_split_metrics", "needs_promotion_gates", "needs_manifest", "blocked_missing_artifacts", "blocked_failed_run"}:
            add_caveat(str(status), "blocker" if str(status).startswith("blocked") else "review", f"{count} campaign row(s) classified as {status}.")

for non_claim in [
    "no_strategy_approval_claim",
    "no_alpha_claim",
    "no_production_readiness_claim",
    "no_statistical_significance_claim",
    "no_promotion_grade_claim",
    "no_complete_platform_artifact_claim_unless_verified",
    "no_ci_runtime_equivalence_claim",
    "no_campaign_correctness_claim_without_native_artifacts",
]:
    add_caveat("non_claim", "info", non_claim)

if "native_campaign_smoke_result" in globals():
    validation = native_campaign_smoke_result.get("provisional_config_validation") or {}
    if native_campaign_smoke_result.get("provisional_config_execution_allowed"):
        add_caveat(
            "provisional_campaign_smoke_config_execution_allowed",
            "review",
            "Generated provisional campaign smoke config is allowed for guarded execution; prefer native template or known-good config when available.",
        )
    if native_campaign_smoke_result.get("status") == "native_campaign_smoke_blocked_provisional_config_validation_failed":
        add_caveat(
            "provisional_campaign_smoke_config_validation_failed",
            "blocker",
            f"Generated provisional campaign smoke config failed local validation: {validation.get('errors', [])}",
        )
    elif validation and validation.get("warnings"):
        add_caveat(
            "provisional_campaign_smoke_config_validation_warning",
            "review",
            f"Generated provisional campaign smoke config validation warnings: {validation.get('warnings', [])}",
        )


if "native_campaign_smoke_result" in globals():
    smoke_status_for_caveat = native_campaign_smoke_result.get("status")
    if smoke_status_for_caveat == "native_campaign_smoke_blocked_dry_run_unavailable":
        add_caveat(
            "native_campaign_smoke_dry_run_unavailable",
            "review",
            "Validated campaign smoke config was available, but the installed stratlake-run-research-campaign help did not advertise a recognized dry-run option."
        )
    if smoke_status_for_caveat == "native_campaign_smoke_blocked_non_dry_run_execution_not_allowed":
        add_caveat(
            "native_campaign_smoke_non_dry_run_blocked",
            "blocker",
            "Native campaign smoke would require non-dry-run execution, but non-dry-run execution was not explicitly allowed."
        )

caveat_register_df = pd.DataFrame(caveats, columns=CAVEAT_COLUMNS)
show_df(caveat_register_df, n=75)


## 18. Write Notebook 12 review artifacts

This cell writes raw review artifacts under `NOTEBOOK12_REVIEW_DIR` when safe. These are generated runtime artifacts and should not be committed unless future import instructions explicitly allow generated fixtures.


In [ ]:
def derive_notebook12_handoff_status(summary: dict[str, Any]) -> str:
    """Return a conservative, audit-friendly handoff status for Notebook 12.

    The status is intentionally descriptive rather than promotional. It separates:
    - source-safe preview with discovery disabled;
    - restored loose discovery with no campaign context;
    - strict negative-control filtering with no campaign context;
    - strict restored discovery blocked because no campaign/run filter was supplied;
    - restored review with some artifacts but incomplete platform evidence;
    - future complete evidence review with human watchlist candidates.
    """
    mode = str(summary.get("notebook12_mode", ""))
    source_safe = bool(summary.get("source_safe_defaults_active"))
    discovery_enabled = bool(globals().get("RUN_CAMPAIGN_ARTIFACT_DISCOVERY", False))
    discover_existing = bool(globals().get("DISCOVER_EXISTING_CAMPAIGN_ARTIFACTS", False))
    strict_required = bool(globals().get("REQUIRE_CAMPAIGN_OR_RUN_FILTER_FOR_RESTORED_REVIEW", False))
    campaign_filter = str(globals().get("CAMPAIGN_ID_FILTER", "") or "").strip()
    run_filter = str(globals().get("RUN_ID_FILTER", "") or "").strip()

    campaign_rows = int(summary.get("campaign_review_rows", 0) or 0)
    artifact_rows = int(summary.get("campaign_artifact_rows", 0) or 0)
    candidate_rows = int(summary.get("candidate_campaign_context_rows", 0) or 0)
    watchlist_rows = int(summary.get("ready_for_human_watchlist_review_count", 0) or 0)
    context_loaded = bool(summary.get("campaign_context_loaded"))


    smoke_status = str(summary.get("native_campaign_smoke_status", "") or "")
    smoke_requested = bool(summary.get("native_campaign_smoke_requested"))
    smoke_executed = bool(summary.get("native_campaign_smoke_executed"))
    smoke_returncode = summary.get("native_campaign_smoke_returncode")
    smoke_config_status = str(summary.get("native_campaign_smoke_config_status", "") or "")

    if mode == "campaign_smoke_run" or smoke_requested or smoke_executed:
        if smoke_status == "native_campaign_smoke_preview_ready":
            return "notebook_12_native_campaign_smoke_preview_ready"
        if smoke_status == "native_campaign_smoke_blocked_missing_preview_config":
            return "notebook_12_native_campaign_smoke_blocked_missing_preview_config"
        if smoke_status == "native_campaign_smoke_blocked_missing_executable_config":
            return "notebook_12_native_campaign_smoke_blocked_missing_executable_config"
        if smoke_status in {"native_campaign_smoke_blocked_dry_run_argument_not_advertised", "native_campaign_smoke_blocked_dry_run_unavailable"}:
            return "notebook_12_native_campaign_smoke_blocked_dry_run_unavailable"
        if smoke_status == "native_campaign_smoke_blocked_non_dry_run_execution_not_allowed":
            return "notebook_12_native_campaign_smoke_blocked_non_dry_run_execution_not_allowed"
        if smoke_status == "native_campaign_smoke_blocked_execution_not_allowed":
            return "notebook_12_native_campaign_smoke_blocked_execution_not_allowed"
        if smoke_executed and smoke_returncode == 0 and summary.get("campaign_artifact_rows", 0):
            return "notebook_12_native_campaign_smoke_completed_with_artifact_discovery"
        if smoke_executed and smoke_returncode == 0:
            return "notebook_12_native_campaign_smoke_dry_run_completed"
        if smoke_executed and smoke_returncode not in (None, 0):
            return "notebook_12_native_campaign_smoke_failed"
        if smoke_config_status in {
            "user_supplied_campaign_config_selected",
            "native_campaign_template_selected",
            "notebook12_generated_smoke_config_selected_reference_only",
            "validated_notebook12_generated_smoke_config_selected_for_execution",
            "unvalidated_notebook12_generated_smoke_config_selected_for_execution",
            "provisional_campaign_smoke_config_selected_for_execution",
            "validated_provisional_campaign_smoke_config_selected_for_execution",
            "unvalidated_provisional_campaign_smoke_config_selected_for_execution",
        }:
            return "notebook_12_native_campaign_smoke_preview_ready"

    if mode == "campaign_preview" and not discovery_enabled and not context_loaded:
        return "campaign_preview_completed_no_runtime_campaign_artifacts_loaded"

    if mode == "restored_campaign_review" and discovery_enabled and discover_existing:
        if strict_required and not (campaign_filter or run_filter):
            return "notebook_12_strict_discovery_blocked_missing_filter"
        if strict_required and (campaign_filter or run_filter) and not context_loaded and artifact_rows == 0 and candidate_rows == 0:
            return "notebook_12_strict_campaign_filter_negative_control_passed"
        if not context_loaded and artifact_rows == 0 and candidate_rows == 0:
            return "notebook_12_restored_campaign_discovery_completed_with_missing_campaign_context"
        if context_loaded and campaign_rows == 0:
            return "notebook_12_restored_campaign_context_inventory_completed_without_review_rows"
        if context_loaded and watchlist_rows > 0:
            return "notebook_12_campaign_evidence_review_completed_with_human_watchlist_candidates"
        if context_loaded:
            return "notebook_12_restored_campaign_review_completed_with_missing_platform_evidence"

    if mode == "campaign_evidence_run" and campaign_rows > 0 and watchlist_rows > 0:
        return "notebook_12_campaign_evidence_review_completed_with_human_watchlist_candidates"

    if context_loaded:
        return "notebook_12_restored_campaign_review_completed_with_missing_platform_evidence"

    return "notebook_12_campaign_review_raw_draft"


summary_payload = {
    "notebook12_test_profile": globals().get("NOTEBOOK12_TEST_PROFILE"),
    "notebook12_test_profile_description": NOTEBOOK12_TEST_PROFILES.get(globals().get("NOTEBOOK12_TEST_PROFILE", ""), {}).get("description"),
    "notebook12_mode": NOTEBOOK12_MODE,
    "session_archive_root": STRATLAKE_SESSION_ARCHIVE_ROOT.as_posix(),
    "session_archive_root_discovery_status": SESSION_ARCHIVE_ROOT_DISCOVERY.get("status"),
    "session_archive_root_discovered_count": len(SESSION_ARCHIVE_ROOT_DISCOVERY.get("discovered_roots", [])),
    "native_campaign_command_preview_count": int(len(native_campaign_preview_df)) if "native_campaign_preview_df" in globals() else 0,
    "native_campaign_command_shape_safe_count": int(native_campaign_preview_df.get("shape_safe_for_preview", pd.Series(dtype=bool)).fillna(False).sum()) if "native_campaign_preview_df" in globals() and not native_campaign_preview_df.empty else 0,
    "native_campaign_command_execution_enabled_count": int(native_campaign_preview_df.get("execution_enabled", pd.Series(dtype=bool)).fillna(False).sum()) if "native_campaign_preview_df" in globals() and not native_campaign_preview_df.empty else 0,
    "native_campaign_command_safe_to_execute_count": int(native_campaign_preview_df.get("safe_to_execute", pd.Series(dtype=bool)).fillna(False).sum()) if "native_campaign_preview_df" in globals() and not native_campaign_preview_df.empty else 0,
    # Backward-compatible summary key retained; use native_campaign_command_safe_to_execute_count for new audits.
    "native_campaign_command_safe_preview_count": int(native_campaign_preview_df.get("safe_to_execute", pd.Series(dtype=bool)).fillna(False).sum()) if "native_campaign_preview_df" in globals() and not native_campaign_preview_df.empty else 0,
    "native_campaign_command_detected_arg_count": int(native_campaign_preview_df.get("detected_argument_count", pd.Series(dtype=int)).fillna(0).sum()) if "native_campaign_preview_df" in globals() and not native_campaign_preview_df.empty else 0,
    "native_campaign_command_unclear_surface_count": int(native_campaign_preview_df.get("command_shape_status", pd.Series(dtype=str)).isin(["argument_surface_unclear", "cli_unavailable", "direct_preview_help_not_captured"]).sum()) if "native_campaign_preview_df" in globals() and not native_campaign_preview_df.empty else 0,
    "native_campaign_help_rows": int(len(report_governance_help_df)) if "report_governance_help_df" in globals() else 0,
    "campaign_context_loaded": bool(
        "candidate_campaign_context_df" in globals()
        and not candidate_campaign_context_df.empty
        and (candidate_campaign_context_df.get("coherence_status", pd.Series(dtype=str)) == "candidate_context_reviewable").any()
    ),
    "campaign_artifact_rows": int(len(campaign_artifact_inventory_df)),
    "total_discovered_artifact_rows": int(len(campaign_artifact_inventory_all_df)) if "campaign_artifact_inventory_all_df" in globals() else int(len(campaign_artifact_inventory_df)),
    "native_campaign_artifact_rows": int(len(campaign_artifact_inventory_df)),
    "notebook12_review_artifact_rows": int(len(notebook12_review_artifact_inventory_df)) if "notebook12_review_artifact_inventory_df" in globals() else 0,
    "native_campaign_marker_rows": int(campaign_artifact_inventory_df.get("native_campaign_marker", pd.Series(dtype=bool)).fillna(False).sum()) if "campaign_artifact_inventory_df" in globals() and not campaign_artifact_inventory_df.empty else 0,
    "candidate_campaign_context_rows": int(len(candidate_campaign_context_df)) if "candidate_campaign_context_df" in globals() else 0,
    "candidate_campaign_context_reviewable_count": int((candidate_campaign_context_df.get("coherence_status", pd.Series(dtype=str)) == "candidate_context_reviewable").sum()) if "candidate_campaign_context_df" in globals() and not candidate_campaign_context_df.empty else 0,
    "campaign_artifact_discovery_root_count": len(CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS) if "CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS" in globals() else 0,
    "campaign_artifact_existing_discovery_root_count": len([root for root in CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS if root.exists()]) if "CAMPAIGN_ARTIFACT_DISCOVERY_ROOTS" in globals() else 0,
    "campaign_candidate_rows": int(len(campaign_evidence_df)),
    "campaign_review_rows": int(len(campaign_evidence_df)),
    "ready_for_human_watchlist_review_count": int((campaign_evidence_df.get("evidence_status", pd.Series(dtype=str)) == "ready_for_human_watchlist_review").sum()) if not campaign_evidence_df.empty else 0,
    "needs_more_evidence_count": int((campaign_evidence_df.get("evidence_status", pd.Series(dtype=str)) == "needs_more_evidence").sum()) if not campaign_evidence_df.empty else 0,
    "blocked_count": int(campaign_evidence_df.get("evidence_status", pd.Series(dtype=str)).astype(str).str.startswith("blocked").sum()) if not campaign_evidence_df.empty else 0,
    "missing_split_metrics_count": int((campaign_evidence_df.get("evidence_status", pd.Series(dtype=str)) == "needs_split_metrics").sum()) if not campaign_evidence_df.empty else 0,
    "missing_promotion_gates_count": int((campaign_evidence_df.get("evidence_status", pd.Series(dtype=str)) == "needs_promotion_gates").sum()) if not campaign_evidence_df.empty else 0,
    "missing_manifest_count": int((campaign_evidence_df.get("evidence_status", pd.Series(dtype=str)) == "needs_manifest").sum()) if not campaign_evidence_df.empty else 0,
    "governance_loaded_count": int(campaign_evidence_df.get("governance_loaded", pd.Series(dtype=bool)).fillna(False).sum()) if not campaign_evidence_df.empty else 0,
    "caveat_count": int(len(caveat_register_df)),
    "promotion_grade_claim_made": False,
    "native_campaign_smoke_requested": bool(native_campaign_smoke_result.get("run_requested", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_executed": bool(native_campaign_smoke_result.get("executed", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_status": native_campaign_smoke_result.get("status") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_config_status": native_campaign_smoke_result.get("config_status") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_config_source": native_campaign_smoke_result.get("config_source") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_config_is_native_template": bool(native_campaign_smoke_result.get("config_is_native_template", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_config_is_notebook_generated": bool(native_campaign_smoke_result.get("config_is_notebook_generated", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_config_validation_status": native_campaign_smoke_result.get("config_validation_status") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_native_template_candidate_count": int(native_campaign_smoke_result.get("native_template_candidate_count", 0)) if "native_campaign_smoke_result" in globals() else 0,
    "native_campaign_smoke_notebook_generated_config_candidate_count": int(native_campaign_smoke_result.get("notebook_generated_smoke_config_candidate_count", 0)) if "native_campaign_smoke_result" in globals() else 0,
    "native_campaign_smoke_dry_run_support_status": native_campaign_smoke_result.get("dry_run_support_status") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_dry_run_supported": bool(native_campaign_smoke_result.get("dry_run_supported", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_dry_run_selected_option": native_campaign_smoke_result.get("dry_run_selected_option") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_allow_non_dry_run_execution": bool(native_campaign_smoke_result.get("allow_non_dry_run_execution", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_provisional_config_execution_allowed": bool(native_campaign_smoke_result.get("provisional_config_execution_allowed", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_provisional_config_validation_required": bool(native_campaign_smoke_result.get("provisional_config_validation_required", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_provisional_config_validation_status": native_campaign_smoke_result.get("config_validation_status") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_provisional_config_validation_valid": (native_campaign_smoke_result.get("provisional_config_validation") or {}).get("valid") if "native_campaign_smoke_result" in globals() and native_campaign_smoke_result.get("provisional_config_validation") is not None else None,
    "native_campaign_smoke_provisional_config_validation_errors": (native_campaign_smoke_result.get("provisional_config_validation") or {}).get("errors") if "native_campaign_smoke_result" in globals() and native_campaign_smoke_result.get("provisional_config_validation") is not None else None,
    "native_campaign_smoke_template_candidates": int(native_campaign_smoke_result.get("template_candidates", 0)) if "native_campaign_smoke_result" in globals() else 0,
    "native_campaign_smoke_selected_config_path": native_campaign_smoke_result.get("selected_config_path") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_preview_config_path": native_campaign_smoke_result.get("preview_config_path") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_executable_config_path": native_campaign_smoke_result.get("executable_config_path") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_can_build_preview": bool(native_campaign_smoke_result.get("can_build_preview", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_can_execute": bool(native_campaign_smoke_result.get("can_execute", False)) if "native_campaign_smoke_result" in globals() else False,
    "native_campaign_smoke_command_preview": native_campaign_smoke_result.get("command_preview") if "native_campaign_smoke_result" in globals() else None,
    "native_campaign_smoke_returncode": native_campaign_smoke_result.get("returncode") if "native_campaign_smoke_result" in globals() else None,
    "runtime_execution_claim_made": bool(native_campaign_smoke_result.get("executed", False)) if "native_campaign_smoke_result" in globals() else NOTEBOOK12_MODE == "campaign_evidence_run",
    "source_safe_defaults_active": SOURCE_SAFE_DEFAULTS_ACTIVE,
    "handoff_status": None,
    "created_at_utc": utc_now_iso(),
}

summary_payload["handoff_status"] = derive_notebook12_handoff_status(summary_payload)

if WRITE_NOTEBOOK12_ARTIFACTS:
    ensure_dir(NOTEBOOK12_REVIEW_DIR)
    (NOTEBOOK12_REVIEW_DIR / "summary.json").write_text(safe_json_dumps(summary_payload), encoding="utf-8")
    if "native_campaign_smoke_result" in globals():
        (NOTEBOOK12_REVIEW_DIR / "native_campaign_smoke_result.json").write_text(safe_json_dumps(native_campaign_smoke_result), encoding="utf-8")
    campaign_artifact_inventory_df.to_csv(NOTEBOOK12_REVIEW_DIR / "campaign_artifact_inventory.csv", index=False)
    if "candidate_campaign_context_df" in globals():
        candidate_campaign_context_df.to_csv(NOTEBOOK12_REVIEW_DIR / "candidate_campaign_context_inventory.csv", index=False)
    campaign_evidence_df.to_csv(NOTEBOOK12_REVIEW_DIR / "campaign_evidence_review.csv", index=False)
    caveat_register_df.to_csv(NOTEBOOK12_REVIEW_DIR / "caveat_register.csv", index=False)

print_json(summary_payload)


## 19. Optional archive checkpoint

Archive checkpointing is off by default. This cell previews the command only unless explicitly enabled.


In [ ]:
CHECKPOINT_ARCHIVE_ID = "notebook-12-campaign-evidence-review"

checkpoint_command = [
    "stratlake-session-archive-bootstrap",
    "--root", str(STRATLAKE_ROOT),
    "--archive-id", CHECKPOINT_ARCHIVE_ID,
    "--archive-collision-policy", "overwrite_allowed",
    "--drive-root", STRATLAKE_SESSION_ARCHIVE_ROOT.as_posix(),
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

checkpoint_result = {
    "command_preview": " ".join(shlex.quote(part) for part in checkpoint_command),
    "run_requested": RUN_STRATLAKE_ARCHIVE_CHECKPOINT,
    "status": "checkpoint_preview_only",
}

if RUN_STRATLAKE_ARCHIVE_CHECKPOINT:
    if command_available(checkpoint_command[0]):
        try:
            result = subprocess.run(checkpoint_command, capture_output=True, text=True, timeout=300, check=False)
            checkpoint_result.update({
                "returncode": result.returncode,
                "stdout_head": result.stdout[:2000],
                "stderr_head": result.stderr[:2000],
                "status": "stratlake_archive_checkpoint_completed" if result.returncode == 0 else "stratlake_archive_checkpoint_returned_nonzero",
            })
        except Exception as exc:
            checkpoint_result.update({"status": "stratlake_archive_checkpoint_error", "error": repr(exc)})
    else:
        checkpoint_result.update({"status": "stratlake_archive_checkpoint_cli_unavailable"})

print_json(checkpoint_result)


## 20. Final handoff

The final handoff summarizes the source/runtime posture, campaign artifact counts, evidence status counts, caveats, non-claims, and recommended next action.


In [ ]:
def campaign_smoke_readiness_reason(summary: dict[str, Any]) -> str:
    """Return a compact, source-safe explanation for campaign-smoke readiness.

    This is deliberately descriptive. It does not claim campaign correctness,
    strategy validity, promotion readiness, or production readiness.
    """
    status = str(summary.get("native_campaign_smoke_status", "") or "")
    config_source = str(summary.get("native_campaign_smoke_config_source", "") or "")
    dry_run_supported = bool(summary.get("native_campaign_smoke_dry_run_supported", False))
    can_build_preview = bool(summary.get("native_campaign_smoke_can_build_preview", False))
    can_execute = bool(summary.get("native_campaign_smoke_can_execute", False))
    executed = bool(summary.get("native_campaign_smoke_executed", False))

    if executed and summary.get("native_campaign_smoke_returncode") == 0:
        return "native_campaign_smoke_executed_successfully_under_enabled_runtime_controls"
    if executed and summary.get("native_campaign_smoke_returncode") not in (None, 0):
        return "native_campaign_smoke_executed_and_failed_under_enabled_runtime_controls"
    if status == "native_campaign_smoke_preview_ready":
        return "command_preview_ready_execution_not_requested_or_not_allowed"
    if status == "native_campaign_smoke_blocked_missing_preview_config":
        return "no_preview_config_available"
    if status == "native_campaign_smoke_blocked_missing_executable_config":
        if config_source == "notebook12_generated_smoke_config":
            return "notebook12_generated_config_available_for_preview_but_not_allowed_as_executable"
        return "no_trusted_native_or_user_supplied_executable_campaign_config_available"
    if status == "native_campaign_smoke_blocked_dry_run_unavailable":
        return "executable_config_available_but_required_dry_run_option_not_advertised"
    if status == "native_campaign_smoke_blocked_non_dry_run_execution_not_allowed":
        return "non_dry_run_execution_blocked_by_explicit_safety_control"
    if status == "native_campaign_smoke_blocked_provisional_config_validation_failed":
        return "generated_or_provisional_config_failed_notebook12_static_validation"
    if can_build_preview and not can_execute:
        return "preview_available_but_execution_not_enabled"
    if can_execute and not dry_run_supported:
        return "execution_candidate_available_but_dry_run_support_not_detected"
    return "campaign_smoke_not_requested_or_preview_only"


def recommend_next_actions(summary: dict[str, Any]) -> list[str]:
    """Return status-specific next actions without over-promoting runtime readiness."""
    actions: list[str] = []
    mode = str(summary.get("notebook12_mode", "") or "")
    smoke_status = str(summary.get("native_campaign_smoke_status", "") or "")
    config_source = str(summary.get("native_campaign_smoke_config_source", "") or "")
    context_loaded = bool(summary.get("campaign_context_loaded"))

    if smoke_status in {"native_campaign_smoke_preview_ready", "native_campaign_smoke_preview_only"}:
        actions.append("review_native_campaign_smoke_command_preview")
        actions.append("run_campaign_smoke_dry_run_profile")

    if smoke_status == "native_campaign_smoke_blocked_missing_preview_config":
        actions.append("prepare_native_campaign_smoke_config")

    if smoke_status == "native_campaign_smoke_blocked_missing_executable_config":
        if config_source == "notebook12_generated_smoke_config":
            actions.append("run_campaign_smoke_dry_run_allow_provisional_profile")
            actions.append("supply_true_native_campaign_template_if_available")
        else:
            actions.append("supply_native_campaign_template_or_user_supplied_campaign_config")
        actions.append("inspect_stratlake_run_research_campaign_help_for_dry_run_support")

    if smoke_status == "native_campaign_smoke_blocked_provisional_config_validation_failed":
        actions.append("fix_or_replace_provisional_campaign_smoke_config")

    if smoke_status in {"native_campaign_smoke_blocked_dry_run_argument_not_advertised", "native_campaign_smoke_blocked_dry_run_unavailable"}:
        actions.append("inspect_stratlake_run_research_campaign_help_for_dry_run_support")
        actions.append("decide_whether_to_allow_explicit_non_dry_run_campaign_smoke")
        actions.append("prefer_true_native_campaign_template_before_non_dry_run_execution")

    if smoke_status == "native_campaign_smoke_blocked_non_dry_run_execution_not_allowed":
        actions.append("enable_explicit_non_dry_run_campaign_smoke_only_if_ready")

    if smoke_status == "native_campaign_smoke_blocked_execution_not_allowed":
        actions.append("enable_guarded_native_campaign_smoke_when_ready")

    # Campaign-context actions are useful, but in campaign-smoke modes they should not
    # obscure the specific smoke-control decision that caused the current handoff.
    if not context_loaded and mode != "campaign_smoke_run":
        actions.append("restore_campaign_archive")
        actions.append("run_campaign_with_native_stratlake_cli")
    elif not context_loaded and mode == "campaign_smoke_run" and smoke_status not in {
        "native_campaign_smoke_blocked_missing_executable_config",
        "native_campaign_smoke_blocked_dry_run_argument_not_advertised",
        "native_campaign_smoke_blocked_dry_run_unavailable",
    }:
        actions.append("restore_campaign_archive_after_smoke_controls_are_validated")

    if summary.get("missing_split_metrics_count", 0) > 0:
        actions.append("add_platform_split_metrics")
    if summary.get("missing_promotion_gates_count", 0) > 0:
        actions.append("add_platform_promotion_gates")
    if summary.get("campaign_review_rows", 0) > 0:
        actions.append("rerun_campaign_evidence_review")
    if summary.get("ready_for_human_watchlist_review_count", 0) > 0:
        actions.append("prepare_human_watchlist_dossier")

    # Preserve order while removing duplicates.
    deduped_actions: list[str] = []
    seen: set[str] = set()
    for action in actions:
        if action not in seen:
            deduped_actions.append(action)
            seen.add(action)

    if not deduped_actions:
        deduped_actions.append("rerun_campaign_evidence_review")
    return deduped_actions


campaign_smoke_execution_readiness = {
    "status": summary_payload.get("native_campaign_smoke_status"),
    "reason": campaign_smoke_readiness_reason(summary_payload),
    "can_build_preview": summary_payload.get("native_campaign_smoke_can_build_preview"),
    "can_execute": summary_payload.get("native_campaign_smoke_can_execute"),
    "executed": summary_payload.get("native_campaign_smoke_executed"),
    "returncode": summary_payload.get("native_campaign_smoke_returncode"),
    "dry_run_support_status": summary_payload.get("native_campaign_smoke_dry_run_support_status"),
    "dry_run_supported": summary_payload.get("native_campaign_smoke_dry_run_supported"),
    "allow_non_dry_run_execution": summary_payload.get("native_campaign_smoke_allow_non_dry_run_execution"),
}

final_handoff = {
    "notebook": "Notebook 12 — StratLake Campaign Evidence Gap and Promotion Readiness Review",
    "raw_stance": "notebook_12_campaign_review_raw_draft",
    "theme": "From isolated expanded runs to campaign-level evidence review.",
    "source_runtime_posture": {
        "notebook12_test_profile": globals().get("NOTEBOOK12_TEST_PROFILE"),
        "notebook12_test_profile_description": NOTEBOOK12_TEST_PROFILES.get(globals().get("NOTEBOOK12_TEST_PROFILE", ""), {}).get("description"),
        "notebook12_mode": NOTEBOOK12_MODE,
        "source_safe_defaults_active": SOURCE_SAFE_DEFAULTS_ACTIVE,
        "runtime_execution_claim_made": summary_payload["runtime_execution_claim_made"],
    },
    "campaign_context_loaded": summary_payload["campaign_context_loaded"],
    "campaign_artifact_rows": summary_payload["campaign_artifact_rows"],
    "native_campaign_artifact_rows": summary_payload.get("native_campaign_artifact_rows", summary_payload["campaign_artifact_rows"]),
    "notebook12_review_artifact_rows": summary_payload.get("notebook12_review_artifact_rows", 0),
    "total_discovered_artifact_rows": summary_payload.get("total_discovered_artifact_rows", summary_payload["campaign_artifact_rows"]),
    "native_campaign_marker_rows": summary_payload.get("native_campaign_marker_rows", 0),
    "candidate_campaign_context_rows": summary_payload.get("candidate_campaign_context_rows", 0),
    "candidate_campaign_context_reviewable_count": summary_payload.get("candidate_campaign_context_reviewable_count", 0),
    "campaign_review_rows": summary_payload["campaign_review_rows"],
    "native_campaign_smoke_config": {
        "source": summary_payload.get("native_campaign_smoke_config_source"),
        "is_native_template": summary_payload.get("native_campaign_smoke_config_is_native_template"),
        "is_notebook_generated": summary_payload.get("native_campaign_smoke_config_is_notebook_generated"),
        "status": summary_payload.get("native_campaign_smoke_config_status"),
        "validation_status": summary_payload.get("native_campaign_smoke_config_validation_status"),
        "selected_config_path": summary_payload.get("native_campaign_smoke_selected_config_path"),
        "preview_config_path": summary_payload.get("native_campaign_smoke_preview_config_path"),
        "executable_config_path": summary_payload.get("native_campaign_smoke_executable_config_path"),
    },
    "native_campaign_smoke_execution_readiness": campaign_smoke_execution_readiness,
    "native_campaign_smoke_dry_run": {
        "support_status": summary_payload.get("native_campaign_smoke_dry_run_support_status"),
        "supported": summary_payload.get("native_campaign_smoke_dry_run_supported"),
        "selected_option": summary_payload.get("native_campaign_smoke_dry_run_selected_option"),
        "allow_non_dry_run_execution": summary_payload.get("native_campaign_smoke_allow_non_dry_run_execution"),
    },
    "evidence_status_counts": evidence_status_counts,
    "caveat_count": summary_payload["caveat_count"],
    "non_claims": [
        "no_strategy_approval_claim",
        "no_alpha_claim",
        "no_production_readiness_claim",
        "no_statistical_significance_claim",
        "no_promotion_grade_claim",
        "no_complete_platform_artifact_claim_unless_verified",
        "no_ci_runtime_equivalence_claim",
        "no_campaign_correctness_claim_without_native_artifacts",
    ],
    "recommended_next_actions": recommend_next_actions(summary_payload),
    "handoff_status": summary_payload["handoff_status"],
}

if WRITE_NOTEBOOK12_ARTIFACTS:
    (NOTEBOOK12_REVIEW_DIR / "final_handoff.json").write_text(safe_json_dumps(final_handoff), encoding="utf-8")

print_json(final_handoff)


## 21. Cold-smoke audit summary and planning-forward posture

Cold smoke testing established that Notebook 12 can run safely without fabricating campaign evidence:

- Cold Smoke 1: baseline preview completed with expected missing artifacts.
- Cold Smoke 2: loose restored discovery completed with missing campaign context.
- Cold Smoke 3: strict missing-campaign negative control passed.
- Cold Smoke 4: strict missing-filter guardrail passed.
- Cold Smoke 5: command-shape readiness regression passed.

The current draft keeps those guardrails and adds the campaign-running bridge:

1. discover native campaign config templates;
2. write a provisional reference config for human editing;
3. construct a native `stratlake-run-research-campaign --config ...` preview;
4. block execution unless explicitly enabled;
5. keep generated provisional configs reference-only unless explicitly allowed;
6. validate generated provisional configs before treating them as executable;
7. require dry-run support by default before running;
8. record dry-run help diagnostics and recognized dry-run option candidates;
9. distinguish true native campaign templates from Notebook 12-generated smoke configs;
10. block non-dry-run campaign smoke unless an explicit non-dry-run override/profile is selected;
11. feed any resulting artifacts back into the campaign evidence discovery/review path.

Recommended next posture after successful cold smokes:

```text
notebook_12_cold_smoke_ready_for_guarded_native_campaign_smoke
```

Recommended progression:

```text
prepare_native_campaign_smoke_config
preview_native_campaign_command
run_native_campaign_smoke_dry_run_or_validated_provisional_dry_run
inspect_dry_run_support_or_explicitly_allow_non_dry_run_smoke
discover_post_run_campaign_artifacts
strict_filter_campaign_id
build_campaign_evidence_review
```

The notebook still does not make approval, alpha, production-readiness, statistical-significance, or promotion-grade claims.

### Smoke-test profile quick switch

For repeat testing, change only this line in the runtime-controls cell:

```python
NOTEBOOK12_TEST_PROFILE = "cold_smoke_1_preview"
```

Common switches:

```text
cold_smoke_1_preview                         # baseline source-safe preview
cold_smoke_2_loose_restored_discovery        # restored artifact discovery, no filter required
cold_smoke_3_strict_negative_control         # synthetic missing campaign id
cold_smoke_4_strict_missing_filter_guardrail # strict mode with no filter supplied
cold_smoke_5_command_shape_readiness         # CLI shape regression check
campaign_smoke_preview                       # prepare/preview native campaign smoke command
campaign_smoke_dry_run                       # guarded opt-in dry-run execution only
campaign_smoke_dry_run_allow_provisional     # validated provisional config, still dry-run only
campaign_smoke_execute_allow_provisional_no_dry_run # explicit non-dry-run smoke; use deliberately
custom                                       # use CUSTOM_RUNTIME_CONTROL_OVERRIDES
```

The profile selector does not weaken execution gates. Native campaign smoke execution still requires the profile/controls to allow it, and the dry-run profiles still block execution if the installed StratLake command does not advertise a recognized dry-run option. Non-dry-run smoke execution is only available through an explicitly named profile or custom override and remains non-claiming.


## 22. Later importation reminder

Suggested future M15 issue sequence:

```text
#117 — M15.1 — Stage and Clean Notebook 12 Campaign Evidence Review
#118 — M15.2 — Classify Notebook 12 Campaign, Restore, Evidence, Governance, and Artifact Surfaces
#119 — M15.3 — Add Notebook 12 Static and Source-Only Campaign Readiness Coverage
#120 — M15.4 — Record Notebook 12 Import Audit, Index, Docs, and Campaign Caveats
#121 — M15.5 — Run Notebook 12 Campaign Runtime Smoke and Evidence Discovery Checks
#122 — M15.6 — Finalize Notebook 12 Campaign Import Handoff and PR Readiness
```

Expected raw source stance:

```text
notebook_12_campaign_review_raw_draft
```

Expected future cleaned stance:

```text
notebook_12_campaign_review_source_safe
```

Expected runtime smoke with missing campaign artifacts:

```text
notebook_12_campaign_preview_completed_with_expected_missing_artifacts
```

Expected loose restored-discovery smoke with no campaign context:

```text
notebook_12_restored_campaign_discovery_completed_with_missing_campaign_context
```

Expected strict missing-campaign negative-control smoke:

```text
notebook_12_strict_campaign_filter_negative_control_passed
```

Expected strict restored-discovery guardrail with no campaign/run filter supplied:

```text
notebook_12_strict_discovery_blocked_missing_filter
```

Expected restored runtime with incomplete evidence:

```text
notebook_12_restored_campaign_review_completed_with_missing_platform_evidence
```

Expected future complete evidence:

```text
notebook_12_campaign_evidence_review_completed_with_human_watchlist_candidates
```


Additional guarded campaign smoke statuses introduced in this draft:

```text
notebook_12_native_campaign_smoke_preview_ready
notebook_12_native_campaign_smoke_blocked_missing_preview_config
notebook_12_native_campaign_smoke_blocked_execution_not_allowed
notebook_12_native_campaign_smoke_blocked_missing_executable_config
notebook_12_native_campaign_smoke_blocked_provisional_config_validation_failed
notebook_12_native_campaign_smoke_blocked_dry_run_unavailable
notebook_12_native_campaign_smoke_blocked_non_dry_run_execution_not_allowed
notebook_12_native_campaign_smoke_dry_run_completed
notebook_12_native_campaign_smoke_completed_with_artifact_discovery
notebook_12_native_campaign_smoke_failed
```


Native campaign artifact filter patch stance:

```text
notebook_12_native_campaign_artifact_filter
```

Notebook 12-generated review artifacts are discoverable for audit but excluded from native campaign context unless real native campaign markers are present.


Smoke config source precision patch stance:

```text
notebook_12_smoke_config_source_precision
```

Notebook 12 now distinguishes true native campaign templates from generated Notebook 12 smoke configs and reports config-source/validation fields separately.
